# Construccion del Universo Final + Variables — Modelo de Fuga NBC
**Etapa: ensamblado de features (RCC, Izipay, scoring, planillas, gestiones) y base final**

Flujo: este notebook arma `hm_kbr_base_modelo_fuga` uniendo el universo+target (`hm_modelo_fuga_bn`) con las tablas de features. **Todas las features son de meses anteriores al corte (as-of t-1), sin leakage.**

Secciones:
1. Configuracion e imports · 2. Utilidades (generate_table) · 3. Tablas de variables (Izipay, Riesgos/Scoring, RCC, IBK, Planillas, Gestiones VPConnect, etc.) · 4. Universo final (todas las variables) · 5. Universo reducido

In [ ]:
# Instalacion consolidada de dependencias
%pip install -q awswrangler==3.13.0 jupyterlab_execute_time seaborn boto3 pandas numpy

# **CONSTRUCCION DE UNIVERSO FINAL + VARIABLES**

In [ ]:
# Nativos
from dateutil.relativedelta import relativedelta
from time import gmtime, strftime
from datetime import datetime 
import random as rn
import joblib
import time
import json
import sys
import os
import gc

#nube
from sagemaker.processing import ProcessingInput, ProcessingOutput
from sagemaker.sklearn.processing import SKLearnProcessor
from sagemaker import get_execution_role
import awswrangler as wr
import sagemaker
import boto3

#calculo
import pandas as pd
import numpy as np
import scipy

#grafico
from IPython.display import display
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline
sns.set(style="whitegrid")

#Interacciones con output
import warnings
warnings.filterwarnings("ignore")
# warnings.simplefilter(action='ignore', category=FutureWarning)
pd.set_option('display.max_rows', 500)
pd.set_option('display.max_columns', 500)
pd.set_option('display.width', 1000)

from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

gc.collect()
# MODELS
##from lightgbm import LGBMClassifier

BASE_DIR = os.path.dirname(os.getcwd())
if BASE_DIR not in sys.path: sys.path.append(BASE_DIR)
    
BASE_DIR = os.path.dirname(os.path.dirname(os.getcwd()))
if BASE_DIR not in sys.path: sys.path.append(BASE_DIR)
print("BASE_DIR::: ", BASE_DIR)
#import scorecardpy as sc

SEED = 123
os.environ['PYTHONHASHSEED'] = str(SEED)
np.random.seed(SEED)
rn.seed(SEED)


from utils_campania import *

In [ ]:
now, region, bucket, grupo_vpc, path_

In [ ]:
bucket_s3 = s3.Bucket(bucket)
bucket_s3

In [ ]:
def generate_table(esquema_vpc, query_, table, path, llave_, 
                   formato='PARQUET', compresion='SNAPPY', grupo=grupo_vpc):
    # seteado de variables
    table_ = table#+'_sg'
    #path=path+'/'+ table_
    path_ = path.replace(table, table_)
    path_del = path_.split(bucket)[-1][1:]
    
    if path_.count('/') <= 6:
        print("ERROR!, ", 'path coincide con uri vpc - PROCESO DETENIDO')
        return {}
    
    if path_.count('/') <= 8:
        print("WARNING!, ", 'puede que la ruta no esté completa')
    
    
    # se elimina la ubicacion fisica
    bucket_s3.objects.filter(Prefix=path_del).delete()
    print(table_, path_, path_del)

    if wr.catalog.does_table_exist(database=esquema_vpc, table=table_):
        # se elimina la tabla en athena
        del_={}
        try:
            del_= wr.catalog.delete_table_if_exists(database=esquema_vpc, table=table_)
        except Exception as e:
            print('e:::::: ', str(e))
        time.sleep(5)
        # se crea nuevamente la tabla
        result_ = wr.athena.create_ctas_table(
            sql=query_,
            database=esquema_vpc,
            ctas_table=table_,
            wait=True,
            s3_output=path_,
            storage_format=formato,
            write_compression=compresion,
            partitioning_info=[llave_],
            athena_query_wait_polling_delay=0.5,
            boto3_session=boto3.Session(),
            workgroup=grupo)
    else:
        del_ = False
        time.sleep(5)
        # se crea nuevamente la tabla
        result_ = wr.athena.create_ctas_table(
            sql=query_,
            database=esquema_vpc,
            ctas_table=table_,
            wait=True,
            s3_output=path_,
            storage_format=formato,
            write_compression=compresion,
            partitioning_info=[llave_],
            athena_query_wait_polling_delay=0.5,
            boto3_session=boto3.Session(),
            workgroup=grupo)
    return del_, result_

def apply_create(table='X', path='X', llave='X', query='X'):
    result = generate_table(esquema_vpc, query, table, path, llave)
    return result[0], result[1]['ctas_query_metadata'].raw_payload['Status']
# ['ctas_query_metadata'].raw_payload['Status']
 

# **TABLAS DE VARIABLES**

## IZIPAY

In [ ]:
df = wr.athena.read_sql_query("""
        SELECT periodo,periodo_ejecucion, count(1), count(distinct num_ruc)
        FROM disc_comercial.HM_IZIPAY_VARIABLES_TOTAL M
        GROUP BY periodo,periodo_ejecucion
        ORDER BY periodo DESC
    """,
    database="e_perm_aws",
    ctas_approach=False
)
print(df.shape)
df.head(5)

In [ ]:
df = wr.athena.read_sql_query("""
        SELECT *
        FROM disc_comercial.HM_IZIPAY_VARIABLES_TOTAL limit 10
    """,
    database="e_perm_aws",
    ctas_approach=False
)
# 2. Extraer columnas e imprimirlas sin comillas y con comas
columnas_formateadas = ", ".join(df.columns)
print(columnas_formateadas)

## VARIABLES RIESGOS CLASIFICACION

In [ ]:
df = wr.athena.read_sql_query("""
        SELECT periodo_gestion, count(1),count(distinct num_ruc)
        FROM disc_comercial.HM_FEATURES_CLAS_RIESGO_FINAL M
        GROUP BY periodo_gestion
        ORDER BY periodo_gestion DESC
    """,
    database="e_perm_aws",
    ctas_approach=False
)
print(df.shape)
df.head(5)

In [ ]:
df = wr.athena.read_sql_query("""
        SELECT *
        FROM disc_comercial.HM_FEATURES_CLAS_RIESGO_FINAL limit 10
    """,
    database="e_perm_aws",
    ctas_approach=False
)
# 2. Extraer columnas e imprimirlas sin comillas y con comas
columnas_formateadas = ", ".join(df.columns)
print(columnas_formateadas)

## SALDOS RCC AJUSTADO DEUDA MAX

In [ ]:
df = wr.athena.read_sql_query("""
        SELECT periodo_analisis, count(1),count(distinct num_ruc)
        FROM  e_perm_aws.t_agg_saldo_rcc_col_directa
        GROUP BY periodo_analisis
        ORDER BY periodo_analisis DESC
    """,
    database="e_perm_aws",
    ctas_approach=False
)
print(df.shape)
df.head(4)

In [ ]:
df = wr.athena.read_sql_query("""
        SELECT *
        FROM disc_comercial.HM_BASE_SALDO_VARIABLES_12M_CONSOLIDADA_SG limit 10
    """,
    database="e_perm_aws",
    ctas_approach=False
)
# 2. Extraer columnas e imprimirlas sin comillas y con comas
columnas_formateadas = ", ".join(df.columns)
print(columnas_formateadas)

## SALDOS IBK

In [ ]:
df = wr.athena.read_sql_query("""
        SELECT periodo, count(1),count(distinct num_ruc)
        FROM disc_comercial.HM_SALDO_VPC_IBK_TOTAL 
        GROUP BY periodo
        ORDER BY periodo DESC
    """,
    database="e_perm_aws",
    ctas_approach=False
)
print(df.shape)
df.head(4)

In [ ]:
df = wr.athena.read_sql_query("""
        SELECT *
        FROM disc_comercial.HM_SALDO_VPC_IBK_TOTAL limit 10
    """,
    database="e_perm_aws",
    ctas_approach=False
)
# 2. Extraer columnas e imprimirlas sin comillas y con comas
columnas_formateadas = ", ".join(df.columns)
print(columnas_formateadas)

## PAGO PLANILLAS

In [ ]:
df = wr.athena.read_sql_query("""
        SELECT periodo_objetivo	, count(1),count(distinct nro_doc_ord)
        FROM disc_comercial.HM_PAGO_PLANILLAS_TOTAL  
        GROUP BY periodo_objetivo	
        ORDER BY periodo_objetivo	 DESC
    """,
    database="e_perm_aws",
    ctas_approach=False
)
print(df.shape)
df.head(5)

In [ ]:
df = wr.athena.read_sql_query("""
        SELECT *
        FROM disc_comercial.HM_PAGO_PLANILLAS_TOTAL limit 10
    """,
    database="e_perm_aws",
    ctas_approach=False
)
# 2. Extraer columnas e imprimirlas sin comillas y con comas
columnas_formateadas = ", ".join(df.columns)
print(columnas_formateadas)

## GESTIONES VPCONNECT

In [ ]:
df = wr.athena.read_sql_query("""
        SELECT periodo_gestion	, count(1),count(distinct num_ruc)
        FROM disc_comercial.HM_GESTION_VPCONNECT_HISTORICAS_BN  
        GROUP BY periodo_gestion	
        ORDER BY periodo_gestion	 DESC
    """,
    database="e_perm_aws",
    ctas_approach=False
)
print(df.shape)
df.head(3)

In [ ]:
df = wr.athena.read_sql_query("""
        SELECT *
        FROM disc_comercial.HM_GESTION_VPCONNECT_HISTORICAS_BN limit 10
    """,
    database="e_perm_aws",
    ctas_approach=False
)
# 2. Extraer columnas e imprimirlas sin comillas y con comas
columnas_formateadas = ", ".join(df.columns)
print(columnas_formateadas)

## SALDOS RCC CLIENTES DETALLE ULTIMO MES

In [ ]:
df = wr.athena.read_sql_query("""
        SELECT COD_MES	, count(1), count(distinct num_ruc)
        FROM e_perm_aws.DS_DETALLE_CLIENTE_RCC_VPC 
        WHERE NUM_RUC IN (SELECT DISTINCT num_ruc FROM disc_comercial.hm_modelos_base_cartera_sg)
           AND NUM_RUC IS NOT NULL  AND  NUM_RUC NOT LIKE ''
        GROUP BY COD_MES	
        ORDER BY COD_MES	 DESC
    """,
    database="e_perm_aws",
    ctas_approach=False
)
print(df.shape)
df.head(5)

In [ ]:
df = wr.athena.read_sql_query("""
        SELECT *
        FROM e_perm_aws.DS_DETALLE_CLIENTE_RCC_VPC limit 10
    """,
    database="e_perm_aws",
    ctas_approach=False
)
# 2. Extraer columnas e imprimirlas sin comillas y con comas
columnas_formateadas = ", ".join(df.columns)
print(columnas_formateadas)

## SALDOS RCC VARIACION

In [ ]:
df = wr.athena.read_sql_query("""
        SELECT COD_MES	, count(1), count(distinct num_ruc)
        FROM e_perm_aws.DS_HST_RCC_VPC  
        WHERE NUM_RUC IN (SELECT DISTINCT num_ruc FROM disc_comercial.hm_modelos_base_cartera_sg)
           AND NUM_RUC IS NOT NULL  AND  NUM_RUC NOT LIKE ''
        GROUP BY COD_MES	
        ORDER BY COD_MES	 DESC
    """,
    database="e_perm_aws",
    ctas_approach=False
)
print(df.shape)
df.head(4)

In [ ]:
df = wr.athena.read_sql_query("""
        SELECT *
        FROM e_perm_aws.DS_HST_RCC_VPC limit 10
    """,
    database="e_perm_aws",
    ctas_approach=False
)
# 2. Extraer columnas e imprimirlas sin comillas y con comas
columnas_formateadas = ", ".join(df.columns)
print(columnas_formateadas)

## RCC TENDENCIA Y SI TIENE SALDO VARIOS PRODUCTOS Y MESES

In [ ]:
df = wr.athena.read_sql_query("""
        SELECT COD_MES	, count(1), count(distinct num_ruc)
        FROM e_perm_aws.DS_TENDENCIA_RCC_VPC  
        WHERE NUM_RUC IN (SELECT DISTINCT num_ruc FROM disc_comercial.hm_modelos_base_cartera_sg)
           AND NUM_RUC IS NOT NULL  AND  NUM_RUC NOT LIKE ''
        GROUP BY COD_MES	
        ORDER BY COD_MES	 DESC
    """,
    database="e_perm_aws",
    ctas_approach=False
)
print(df.shape)
df.head(10)

In [ ]:
df = wr.athena.read_sql_query("""
        SELECT *
        FROM e_perm_aws.DS_TENDENCIA_RCC_VPC 
        WHERE NUM_RUC IN (SELECT DISTINCT num_ruc FROM disc_comercial.hm_modelos_base_cartera_sg)
        limit 10
    """,
    database="e_perm_aws",
    ctas_approach=False
)
# 2. Extraer columnas e imprimirlas sin comillas y con comas
columnas_formateadas = ", ".join(df.columns)
print(columnas_formateadas)

## RCC PRINCIPALIDAD SALDO Y TIPO ENTIDAD FINANCIERA

In [ ]:
df = wr.athena.read_sql_query("""
        SELECT COD_MES	, count(1), count(distinct num_ruc)
        FROM e_perm_aws.DS_PRINCIPALIDAD_VPC  
        WHERE NUM_RUC IN (SELECT DISTINCT num_ruc FROM disc_comercial.hm_modelos_base_cartera_sg)
           AND NUM_RUC IS NOT NULL  AND  NUM_RUC NOT LIKE ''
        GROUP BY COD_MES	
        ORDER BY COD_MES	 DESC
    """,
    database="e_perm_aws",
    ctas_approach=False
)
print(df.shape)
df.head(10)

In [ ]:
df = wr.athena.read_sql_query("""
        SELECT *
        FROM e_perm_aws.DS_PRINCIPALIDAD_VPC 
        WHERE NUM_RUC IN (SELECT DISTINCT num_ruc FROM disc_comercial.hm_modelos_base_cartera_sg)
        limit 10
    """,
    database="e_perm_aws",
    ctas_approach=False
)
# 2. Extraer columnas e imprimirlas sin comillas y con comas
columnas_formateadas = ", ".join(df.columns)
print(columnas_formateadas)

## VARIACION SALDOS

In [ ]:
df = wr.athena.read_sql_query("""
        SELECT COD_MES	, count(1), count(distinct num_ruc)
        FROM e_perm_aws.DS_VAR_SALDOS_VPC  
         WHERE NUM_RUC IN (SELECT DISTINCT num_ruc FROM disc_comercial.hm_modelos_base_cartera_sg)
           AND NUM_RUC IS NOT NULL  AND  NUM_RUC NOT LIKE ''
        GROUP BY COD_MES	
        ORDER BY COD_MES	 DESC
    """,
    database="e_perm_aws",
    ctas_approach=False
)
print(df.shape)
df.head(10)

In [ ]:
df = wr.athena.read_sql_query("""
        SELECT *
        FROM e_perm_aws.DS_VAR_SALDOS_VPC 
        WHERE NUM_RUC IN (SELECT DISTINCT num_ruc FROM disc_comercial.hm_modelos_base_cartera_sg)
        limit 10
    """,
    database="e_perm_aws",
    ctas_approach=False
)
# 2. Extraer columnas e imprimirlas sin comillas y con comas
columnas_formateadas = ", ".join(df.columns)
print(columnas_formateadas)

## **UNIVERSO FINAL TODAS LAS VARIABLES**

In [ ]:
table_universo ='hm_kbr_base_modelo_fuga'
apply_create(
    table=table_universo, 
    path='{}vpc_new/FUGA_CREDITO_BN/ATHENA/BASE_MODELO/{}/'.format(path_, table_universo), 
    llave='periodo_ejecucion',
    query="""
    SELECT  
    A.periodo periodo_campania,
    A.numeroruc num_ruc,
    A.y_2m,
    A.y_3m ,

    COALESCE(b.saldo_total,0) saldo_total,
    COALESCE(b.n_creditos,0) n_creditos,
    COALESCE(b.monto_desem_total,0) monto_desem_total,
    COALESCE(b.pct_saldo_remanente,0) pct_saldo_remanente,
    COALESCE(b.tea_principal,0) tea_principal,
    COALESCE(b.meses_desde_desem_min,-1) meses_desde_desem_min,
    COALESCE(b.cuotas_total,0) cuotas_total,
    COALESCE(b.cuotas_pend_total,0) cuotas_pend_total,
    COALESCE(b.pct_avance_cuotas,0) pct_avance_cuotas,
    COALESCE(b.cuota_total,0) cuota_total,
    COALESCE(b.atraso_max,0) atraso_max,
    COALESCE(b.scoring_min,0) scoring_min,
    COALESCE(b.dias_venc_max,0) dias_venc_max,
    COALESCE(b.flg_cap_trabajo,0) flg_cap_trabajo,
    COALESCE(b.flg_compra_deuda,0) flg_compra_deuda,
    COALESCE(b.saldo_cap_trabajo,0) saldo_cap_trabajo,
    COALESCE(b.monto_desem_cap_trabajo,0) monto_desem_cap_trabajo,
    COALESCE(b.tea_cap_trabajo,0) tea_cap_trabajo,
    COALESCE(b.meses_desde_desem_cap_trabajo,-1) meses_desde_desem_cap_trabajo,
    COALESCE(b.cuotas_total_cap_trabajo,0) cuotas_total_cap_trabajo,
    COALESCE(b.cuotas_pend_cap_trabajo,0) cuotas_pend_cap_trabajo,
    COALESCE(b.pct_avance_cuotas_cap_trabajo,0) pct_avance_cuotas_cap_trabajo,
    COALESCE(b.cuota_cap_trabajo,0) cuota_cap_trabajo,
    COALESCE(b.pct_d_saldo_3m_cap_trabajo,0) pct_d_saldo_3m_cap_trabajo,
    COALESCE(b.pct_d_saldo_6m_cap_trabajo,0) pct_d_saldo_6m_cap_trabajo,
    COALESCE(b.saldo_compra_deuda,0) saldo_compra_deuda,
    COALESCE(b.monto_desem_compra_deuda,0) monto_desem_compra_deuda,
    COALESCE(b.tea_pond_compra_deuda,0) tea_pond_compra_deuda,
    COALESCE(b.meses_desde_desem_compra_deuda,-1) meses_desde_desem_compra_deuda,
    COALESCE(b.cuotas_total_compra_deuda,0) cuotas_total_compra_deuda,
    COALESCE(b.cuotas_pend_compra_deuda,0) cuotas_pend_compra_deuda,
    COALESCE(b.pct_avance_cuotas_compra_deuda,0) pct_avance_cuotas_compra_deuda,
    COALESCE(b.cuota_compra_deuda,0) cuota_compra_deuda,
    COALESCE(b.pct_d_saldo_3m_compra_deuda,0) pct_d_saldo_3m_compra_deuda,
    COALESCE(b.pct_d_saldo_6m_compra_deuda,0) pct_d_saldo_6m_compra_deuda,
    COALESCE(b.pct_d_saldo_3m,0) pct_d_saldo_3m,
    COALESCE(b.pct_d_saldo_6m,0) pct_d_saldo_6m,
    COALESCE(b.d_tea_pond_3m,0) d_tea_pond_3m,
    COALESCE(b.d_n_creditos_3m,0) d_n_creditos_3m,
    COALESCE(b.n_prepagos_6m,0) n_prepagos_6m,
    COALESCE(b.monto_prepago_6m,0) monto_prepago_6m,
    COALESCE(b.max_caida_rel_3m,0) max_caida_rel_3m,


    --SUNAT RENIEC
    SR.tiempo_vida_empresa,
    SR.tiempo_ya_no_funciona,
    SR.cant_trabajadores,
    SR.flg_pj,
    SR.tip_contribuyente_val,
    SR.ciiu_val,
    SR.estado_contribuyente,
    SR.productos_totales_rrll,
    SR.saldo_maximo_prom_activo_rrll,
    SR.tenencia_total_rrll,
    SR.tenencia_promedio_rrll,
    SR.tenencia_maxima_rrll,
    SR.producto_promedio_rrll,
    SR.producto_maximo_rrll,
    SR.ingreso_bruto_total_rrll,
    SR.ingreso_bruto_promedio_rrll,
    SR.ingreso_bruto_maximo_rrll,

    --VPCONNECT
    COALESCE(VPC.recencia_gestion_meses,24) recencia_gestion_meses,
    COALESCE(VPC.recencia_acepta_meses, 24) recencia_acepta_meses,
    COALESCE(VPC.recencia_gestion_activa_meses,24) recencia_gestion_activa_meses,
    COALESCE(VPC.recencia_no_acepta_meses,24) recencia_no_acepta_meses,
    COALESCE(VPC.recencia_pensara_meses,24) recencia_pensara_meses,
    COALESCE(VPC.meses_gestionados_ult12m,0) meses_gestionados_ult12m,
    COALESCE(VPC.meses_gestionados_ult6m,0) meses_gestionados_ult6m,
    COALESCE(VPC.meses_gestionados_ult3m,0) meses_gestionados_ult3m,
    COALESCE(VPC.monto_oferta_promedio_ult12m,0) monto_oferta_promedio_ult12m,
    COALESCE(VPC.monto_oferta_max_ult12m,0) monto_oferta_max_ult12m,
    COALESCE(VPC.monto_oferta_min_ult12m,0) monto_oferta_min_ult12m,
    COALESCE(VPC.tasa_promedio_ult12m,999) tasa_promedio_ult12m,
    COALESCE(VPC.tasa_max_ult12m,999) tasa_max_ult12m,
    COALESCE(VPC.tasa_min_ult12m,999) tasa_min_ult12m,
    COALESCE(VPC.monto_oferta_promedio_ult6m,0) monto_oferta_promedio_ult6m,
    COALESCE(VPC.monto_oferta_max_ult6m,0) monto_oferta_max_ult6m,
    COALESCE(VPC.monto_oferta_min_ult6m,0) monto_oferta_min_ult6m,
    COALESCE(VPC.tasa_promedio_ult6m,999) tasa_promedio_ult6m,
    COALESCE(VPC.tasa_max_ult6m,999) tasa_max_ult6m,
    COALESCE(VPC.tasa_min_ult6m,999) tasa_min_ult6m,
    COALESCE(VPC.monto_oferta_promedio_ult3m,0) monto_oferta_promedio_ult3m,
    COALESCE(VPC.monto_oferta_max_ult3m,0) monto_oferta_max_ult3m,
    COALESCE(VPC.monto_oferta_min_ult3m,0) monto_oferta_min_ult3m,
    COALESCE(VPC.tasa_promedio_ult3m,999) tasa_promedio_ult3m,
    COALESCE(VPC.tasa_max_ult3m,999) tasa_max_ult3m,
    COALESCE(VPC.tasa_min_ult3m,999) tasa_min_ult3m,
    COALESCE(VPC.flg_recurrente_agil_ult12m,0) flg_recurrente_agil_ult12m,
    COALESCE(VPC.flg_impulso_ult12m,0) flg_impulso_ult12m,
    COALESCE(VPC.flg_nuevo_aprobado_ult12m,0) flg_nuevo_aprobado_ult12m,
    COALESCE(VPC.flg_nuevo_aprobado_agil_ult12m,0) flg_nuevo_aprobado_agil_ult12m,
    COALESCE(VPC.flg_nuevo_pre_ult12m,0) flg_nuevo_pre_ult12m,
    COALESCE(VPC.flg_otros_recurrente_ult12m,0) flg_otros_recurrente_ult12m,
    COALESCE(VPC.flg_garantia_ult12m,0) flg_garantia_ult12m,
    COALESCE(VPC.flg_compra_deuda_ult12m,0) flg_compra_deuda_ult12m,
    COALESCE(VPC.flg_giro_linea_ult12m,0) flg_giro_linea_ult12m,
    COALESCE(VPC.flg_crecer_ult12m,0) flg_crecer_ult12m,
    COALESCE(VPC.flg_recurrente_agil_ult6m,0) flg_recurrente_agil_ult6m,
    COALESCE(VPC.flg_impulso_ult6m,0) flg_impulso_ult6m,
    COALESCE(VPC.flg_nuevo_aprobado_ult6m,0) flg_nuevo_aprobado_ult6m,
    COALESCE(VPC.flg_nuevo_aprobado_agil_ult6m,0) flg_nuevo_aprobado_agil_ult6m,
    COALESCE(VPC.flg_nuevo_pre_ult6m,0) flg_nuevo_pre_ult6m,
    COALESCE(VPC.flg_otros_recurrente_ult6m,0) flg_otros_recurrente_ult6m,
    COALESCE(VPC.flg_garantia_ult6m,0) flg_garantia_ult6m,
    COALESCE(VPC.flg_compra_deuda_ult6m,0) flg_compra_deuda_ult6m,
    COALESCE(VPC.flg_giro_linea_ult6m,0) flg_giro_linea_ult6m,
    COALESCE(VPC.flg_crecer_ult6m,0) flg_crecer_ult6m,
    COALESCE(VPC.flg_recurrente_agil_ult3m,0) flg_recurrente_agil_ult3m,
    COALESCE(VPC.flg_impulso_ult3m,0) flg_impulso_ult3m,
    COALESCE(VPC.flg_nuevo_aprobado_ult3m,0) flg_nuevo_aprobado_ult3m,
    COALESCE(VPC.flg_nuevo_aprobado_agil_ult3m,0) flg_nuevo_aprobado_agil_ult3m,
    COALESCE(VPC.flg_nuevo_pre_ult3m,0) flg_nuevo_pre_ult3m,
    COALESCE(VPC.flg_otros_recurrente_ult3m,0) flg_otros_recurrente_ult3m,
    COALESCE(VPC.flg_garantia_ult3m,0) flg_garantia_ult3m,
    COALESCE(VPC.flg_compra_deuda_ult3m,0) flg_compra_deuda_ult3m,
    COALESCE(VPC.flg_giro_linea_ult3m,0) flg_giro_linea_ult3m,
    COALESCE(VPC.flg_crecer_ult3m,0) flg_crecer_ult3m,
    COALESCE(VPC.flg_no_acepta_campana_ult12m,0) flg_no_acepta_campana_ult12m,
    COALESCE(VPC.flg_no_califica_ult12m,0) flg_no_califica_ult12m,
    COALESCE(VPC.flg_desistio_ult12m,0) flg_desistio_ult12m,
    COALESCE(VPC.flg_sin_contacto_ult12m,0) flg_sin_contacto_ult12m,
    COALESCE(VPC.flg_pensara_ult12m,0) flg_pensara_ult12m,
    COALESCE(VPC.flg_acepta_campana_ult6m,0) flg_acepta_campana_ult6m,
    COALESCE(VPC.flg_acepta_campana_ult12m,0) flg_acepta_campana_ult12m,
    COALESCE(VPC.flg_acepta_campana_ult3m,0) flg_acepta_campana_ult3m,
    COALESCE(VPC.flg_no_acepta_campana_ult6m,0) flg_no_acepta_campana_ult6m,
    COALESCE(VPC.flg_no_califica_ult6m,0) flg_no_califica_ult6m,
    COALESCE(VPC.flg_desistio_ult6m,0) flg_desistio_ult6m,
    COALESCE(VPC.flg_sin_contacto_ult6m,0) flg_sin_contacto_ult6m,
    COALESCE(VPC.flg_pensara_ult6m,0) flg_pensara_ult6m,
    COALESCE(VPC.flg_no_acepta_campana_ult3m,0) flg_no_acepta_campana_ult3m,
    COALESCE(VPC.flg_no_califica_ult3m,0) flg_no_califica_ult3m,
    COALESCE(VPC.flg_desistio_ult3m,0) flg_desistio_ult3m,
    COALESCE(VPC.flg_sin_contacto_ult3m,0) flg_sin_contacto_ult3m,
    COALESCE(VPC.flg_pensara_ult3m,0) flg_pensara_ult3m,
    COALESCE(VPC.flg_no_acepta_tasa_elevada_ult12m,0) flg_no_acepta_tasa_elevada_ult12m,
    COALESCE(VPC.flg_no_acepta_monto_ult12m,0) flg_no_acepta_monto_ult12m,
    COALESCE(VPC.flg_no_acepta_liquidez_ult12m,0) flg_no_acepta_liquidez_ult12m,
    COALESCE(VPC.flg_no_acepta_otro_banco_ult12m,0) flg_no_acepta_otro_banco_ult12m,
    COALESCE(VPC.flg_no_califica_cem_ult12m,0) flg_no_califica_cem_ult12m,
    COALESCE(VPC.flg_no_califica_sobreendeuda_ult12m,0) flg_no_califica_sobreendeuda_ult12m,
    COALESCE(VPC.flg_no_califica_sbs_ult12m,0) flg_no_califica_sbs_ult12m,
    COALESCE(VPC.flg_desistio_tasa_ult12m,0) flg_desistio_tasa_ult12m,
    COALESCE(VPC.flg_pensara_tasa_ult12m,0) flg_pensara_tasa_ult12m,
    COALESCE(VPC.flg_pensara_monto_ult12m,0) flg_pensara_monto_ult12m,
    COALESCE(VPC.flg_no_acepta_tasa_elevada_ult6m,0) flg_no_acepta_tasa_elevada_ult6m,
    COALESCE(VPC.flg_no_acepta_monto_ult6m,0) flg_no_acepta_monto_ult6m,
    COALESCE(VPC.flg_no_acepta_liquidez_ult6m,0) flg_no_acepta_liquidez_ult6m,
    COALESCE(VPC.flg_no_acepta_otro_banco_ult6m,0) flg_no_acepta_otro_banco_ult6m,
    COALESCE(VPC.flg_no_califica_cem_ult6m,0) flg_no_califica_cem_ult6m,
    COALESCE(VPC.flg_no_califica_sobreendeuda_ult6m,0) flg_no_califica_sobreendeuda_ult6m,
    COALESCE(VPC.flg_no_califica_sbs_ult6m,0) flg_no_califica_sbs_ult6m,
    COALESCE(VPC.flg_desistio_tasa_ult6m,0) flg_desistio_tasa_ult6m,
    COALESCE(VPC.flg_pensara_tasa_ult6m,0) flg_pensara_tasa_ult6m,
    COALESCE(VPC.flg_pensara_monto_ult6m,0) flg_pensara_monto_ult6m,
    COALESCE(VPC.flg_no_acepta_tasa_elevada_ult3m,0) flg_no_acepta_tasa_elevada_ult3m,
    COALESCE(VPC.flg_no_acepta_monto_ult3m,0) flg_no_acepta_monto_ult3m,
    COALESCE(VPC.flg_no_acepta_liquidez_ult3m,0) flg_no_acepta_liquidez_ult3m,
    COALESCE(VPC.flg_no_acepta_otro_banco_ult3m,0) flg_no_acepta_otro_banco_ult3m,
    COALESCE(VPC.flg_no_califica_cem_ult3m,0) flg_no_califica_cem_ult3m,
    COALESCE(VPC.flg_no_califica_sobreendeuda_ult3m,0) flg_no_califica_sobreendeuda_ult3m,
    COALESCE(VPC.flg_no_califica_sbs_ult3m,0) flg_no_califica_sbs_ult3m,
    COALESCE(VPC.flg_desistio_tasa_ult3m,0) flg_desistio_tasa_ult3m,
    COALESCE(VPC.flg_pensara_tasa_ult3m,0) flg_pensara_tasa_ult3m,
    COALESCE(VPC.flg_pensara_monto_ult3m,0) flg_pensara_monto_ult3m,
    COALESCE(VPC.tasa_no_acepta_ult12m,0) tasa_no_acepta_ult12m,
    COALESCE(VPC.tasa_no_califica_ult12m,0) tasa_no_califica_ult12m,
    COALESCE(VPC.tasa_pensara_ult12m,0) tasa_pensara_ult12m,
    COALESCE(VPC.tasa_sin_contacto_ult12m,0) tasa_sin_contacto_ult12m,

    --- PLANILLA 
    COALESCE(PLA.total_planillas_3m,0 ) total_planillas_3m,
    COALESCE(PLA.total_planillas_6m,0 ) total_planillas_6m,
    COALESCE(PLA.total_planillas_u12m,0 ) total_planillas_u12m,
    COALESCE(PLA.total_imp_3m,0 ) total_imp_3m,
    COALESCE(PLA.total_imp_u6m,0 ) total_imp_u6m,
    COALESCE(PLA.total_imp_u12m,0 ) total_imp_u12m,
    COALESCE(PLA.promedio_nro_planillas_3m,0 ) promedio_nro_planillas_3m,
    COALESCE(PLA.promedio_nro_planillas_6m,0 ) promedio_nro_planillas_6m,
    COALESCE(PLA.promedio_nro_planillas_12m,0 ) promedio_nro_planillas_12m,
    COALESCE(PLA.promedio_imp_3m,0 ) promedio_imp_3m,
    COALESCE(PLA.promedio_imp_6m,0 ) promedio_imp_6m,
    COALESCE(PLA.promedio_imp_12m,0 ) promedio_imp_12m,
    COALESCE(PLA.max_nro_planillas_3m,0 ) max_nro_planillas_3m,
    COALESCE(PLA.max_nro_planillas_6m,0 ) max_nro_planillas_6m,
    COALESCE(PLA.max_nro_planillas_12m,0 ) max_nro_planillas_12m,
    COALESCE(PLA.min_nro_planillas_3m,-1 ) min_nro_planillas_3m,
    COALESCE(PLA.min_nro_planillas_6m,-1 ) min_nro_planillas_6m,
    COALESCE(PLA.min_nro_planillas_12m,-1 ) min_nro_planillas_12m,
    COALESCE(PLA.max_imp_3m,0 ) max_imp_3m,
    COALESCE(PLA.max_imp_6m,0 ) max_imp_6m,
    COALESCE(PLA.max_imp_12m,0 ) max_imp_12m,
    COALESCE(PLA.meses_activos_u3m,0 ) meses_activos_u3m,
    COALESCE(PLA.meses_activos_u6m,0 ) meses_activos_u6m,
    COALESCE(PLA.meses_activos_u12m,0 ) meses_activos_u12m,
    COALESCE(PLA.porc_meses_activos_u3m,0 ) porc_meses_activos_u3m,
    COALESCE(PLA.porc_meses_activos_u6m,0 ) porc_meses_activos_u6m,
    COALESCE(PLA.porc_meses_activos_u12m,0 ) porc_meses_activos_u12m,
    COALESCE(PLA.flg_pago_todos_u3m,0 ) flg_pago_todos_u3m,
    COALESCE(PLA.flg_pago_todos_u6m,0 ) flg_pago_todos_u6m,
    COALESCE(PLA.flg_pago_todos_u12m,0 ) flg_pago_todos_u12m,
    COALESCE(PLA.diff_planillas_m1_m2,-999999 ) diff_planillas_m1_m2,
    COALESCE(PLA.diff_planillas_m2_m3,-999999 ) diff_planillas_m2_m3,
    COALESCE(PLA.diff_planillas_m3_m4,-999999 ) diff_planillas_m3_m4,
    COALESCE(PLA.var_porc_m1_m2,-999999 ) var_porc_m1_m2,
    COALESCE(PLA.var_porc_m2_m3,-999999 ) var_porc_m2_m3,
    COALESCE(PLA.var_porc_m3_m4,-999999 ) var_porc_m3_m4,
    COALESCE(PLA.var_porc_3m_vs_3m,-999999 ) var_porc_3m_vs_3m,
    COALESCE(PLA.var_porc_6m_vs_6m,-999999 ) var_porc_6m_vs_6m,
    COALESCE(PLA.var_porc_3m_vs_prom_12m,-999999 ) var_porc_3m_vs_prom_12m,
    COALESCE(PLA.ticket_promedio_3m,0 ) ticket_promedio_3m,
    COALESCE(PLA.ticket_promedio_6m,0 ) ticket_promedio_6m,
    COALESCE(PLA.ticket_promedio_12m,0 ) ticket_promedio_12m,
    COALESCE(PLA.flag_crecimiento_sostenido_3m,0 ) flag_crecimiento_sostenido_3m,
    COALESCE(PLA.flag_decrecimiento_sostenido_3m,0 ) flag_decrecimiento_sostenido_3m,
    COALESCE(PLA.flag_3m_sobre_promedio_12m,0 ) flag_3m_sobre_promedio_12m,
    COALESCE(PLA.flag_mas_1_planilla_3m,0 ) flag_mas_1_planilla_3m,
    COALESCE(PLA.flag_mas_10_planillas_3m,0 ) flag_mas_10_planillas_3m,
    COALESCE(PLA.flag_mas_20_planillas_3m,0 ) flag_mas_20_planillas_3m,
    COALESCE(PLA.flag_mas_50_planillas_3m,0 ) flag_mas_50_planillas_3m,
    COALESCE(PLA.flag_mas_100_planillas_3m,0 ) flag_mas_100_planillas_3m,
    COALESCE(PLA.flag_mas_200_planillas_3m,0 ) flag_mas_200_planillas_3m,
    COALESCE(PLA.flag_mas_500_planillas_3m,0 ) flag_mas_500_planillas_3m,
    COALESCE(PLA.flag_mas_1_planilla_6m,0 ) flag_mas_1_planilla_6m,
    COALESCE(PLA.flag_mas_10_planillas_6m,0 ) flag_mas_10_planillas_6m,
    COALESCE(PLA.flag_mas_20_planillas_6m,0 ) flag_mas_20_planillas_6m,
    COALESCE(PLA.flag_mas_50_planillas_6m,0 ) flag_mas_50_planillas_6m,
    COALESCE(PLA.flag_mas_100_planillas_6m,0 ) flag_mas_100_planillas_6m,
    COALESCE(PLA.flag_mas_200_planillas_6m,0 ) flag_mas_200_planillas_6m,
    COALESCE(PLA.flag_mas_500_planillas_6m,0 ) flag_mas_500_planillas_6m,
    COALESCE(PLA.flag_mas_1_planilla_12m,0 ) flag_mas_1_planilla_12m,
    COALESCE(PLA.flag_mas_10_planillas_12m,0 ) flag_mas_10_planillas_12m,
    COALESCE(PLA.flag_mas_20_planillas_12m,0 ) flag_mas_20_planillas_12m,
    COALESCE(PLA.flag_mas_50_planillas_12m,0 ) flag_mas_50_planillas_12m,
    COALESCE(PLA.flag_mas_100_planillas_12m,0 ) flag_mas_100_planillas_12m,
    COALESCE(PLA.flag_mas_200_planillas_12m,0 ) flag_mas_200_planillas_12m,
    COALESCE(PLA.flag_mas_500_planillas_12m,0 ) flag_mas_500_planillas_12m,
    COALESCE(PLA.flag_mas_1000_planillas_12m,0 ) flag_mas_1000_planillas_12m,
    COALESCE(PLA.meses_desde_ultima_actividad,24 ) meses_desde_ultima_actividad,
    COALESCE(PLA.flag_cliente_activo_3m,0 ) flag_cliente_activo_3m,
    COALESCE(PLA.flag_cliente_en_riesgo,0 ) flag_cliente_en_riesgo,
    COALESCE(PLA.flag_cliente_perdido,0 ) flag_cliente_perdido,
    COALESCE(PLA.flg_tuvo_mas_5_algun_mes_3m,0 ) flg_tuvo_mas_5_algun_mes_3m,
    COALESCE(PLA.flg_tuvo_mas_10_algun_mes_3m,0 ) flg_tuvo_mas_10_algun_mes_3m,
    COALESCE(PLA.flg_tuvo_mas_20_algun_mes_3m,0 ) flg_tuvo_mas_20_algun_mes_3m,
    COALESCE(PLA.flg_tuvo_mas_50_algun_mes_3m,0 ) flg_tuvo_mas_50_algun_mes_3m,
    COALESCE(PLA.flg_tuvo_mas_100_algun_mes_3m,0 ) flg_tuvo_mas_100_algun_mes_3m,
    COALESCE(PLA.flg_tuvo_mas_5_algun_mes_6m,0 ) flg_tuvo_mas_5_algun_mes_6m,
    COALESCE(PLA.flg_tuvo_mas_10_algun_mes_6m,0 ) flg_tuvo_mas_10_algun_mes_6m,
    COALESCE(PLA.flg_tuvo_mas_20_algun_mes_6m,0 ) flg_tuvo_mas_20_algun_mes_6m,
    COALESCE(PLA.flg_tuvo_mas_50_algun_mes_6m,0 ) flg_tuvo_mas_50_algun_mes_6m,
    COALESCE(PLA.flg_tuvo_mas_100_algun_mes_6m,0 ) flg_tuvo_mas_100_algun_mes_6m,
    COALESCE(PLA.flg_tuvo_mas_5_algun_mes_12m,0 ) flg_tuvo_mas_5_algun_mes_12m,
    COALESCE(PLA.flg_tuvo_mas_10_algun_mes_12m,0 ) flg_tuvo_mas_10_algun_mes_12m,
    COALESCE(PLA.flg_tuvo_mas_20_algun_mes_12m,0 ) flg_tuvo_mas_20_algun_mes_12m,
    COALESCE(PLA.flg_tuvo_mas_50_algun_mes_12m,0 ) flg_tuvo_mas_50_algun_mes_12m,
    COALESCE(PLA.flg_tuvo_mas_100_algun_mes_12m,0 ) flg_tuvo_mas_100_algun_mes_12m,
    COALESCE(PLA.flg_nomina_mayor_100k_planillas_12m,0 ) flg_nomina_mayor_100k_planillas_12m,
    COALESCE(PLA.flg_nomina_mayor_50k_planillas_12m,0 ) flg_nomina_mayor_50k_planillas_12m,
    COALESCE(PLA.cnt_meses_nomina_mayor_100k_planillas_12m,0 ) cnt_meses_nomina_mayor_100k_planillas_12m,
    COALESCE(PLA.cnt_meses_nomina_mayor_50k_planillas_12m,0 ) cnt_meses_nomina_mayor_50k_planillas_12m,
    COALESCE(PLA.capacidad_cuota_mensual_planillas,0 ) capacidad_cuota_mensual_planillas,
    COALESCE(PLA.capacidad_credito_estimada_planillas,0 ) capacidad_credito_estimada_planillas,
    COALESCE(PLA.flg_capacidad_mayor_100k_planillas,0 ) flg_capacidad_mayor_100k_planillas,



    --SALDOS IBK 
    COALESCE(SLD.flg_vigente,0 ) flg_vigente,
    COALESCE(SLD.flg_vencido,0 ) flg_vencido,
    COALESCE(SLD.flg_castigado,0 ) flg_castigado,
    COALESCE(SLD.saldo_deteriorado,0 ) saldo_deteriorado,
    COALESCE(SLD.ratio_mora,0 ) ratio_mora,
    COALESCE(SLD.flg_mora_historica_6m,0 ) flg_mora_historica_6m,
    COALESCE(SLD.flg_castigado_historico_6m,0 ) flg_castigado_historico_6m,
    COALESCE(SLD.var_abs_pasivo_1m,-99999999 ) var_abs_pasivo_1m,
    COALESCE(SLD.var_abs_pasivo_3m,-99999999 ) var_abs_pasivo_3m,
    COALESCE(SLD.var_abs_pasivo_6m,-99999999 ) var_abs_pasivo_6m,
    COALESCE(SLD.var_abs_pasivo_12m,-99999999 ) var_abs_pasivo_12m,
    COALESCE(SLD.var_abs_colocaciones_1m,-99999999 ) var_abs_colocaciones_1m,
    COALESCE(SLD.var_abs_colocaciones_3m,-99999999 ) var_abs_colocaciones_3m,
    COALESCE(SLD.var_abs_colocaciones_6m,-99999999 ) var_abs_colocaciones_6m,
    COALESCE(SLD.var_abs_colocaciones_12m,-99999999 ) var_abs_colocaciones_12m,
    COALESCE(SLD.var_pct_pasivo_1m,-9999 ) var_pct_pasivo_1m,
    COALESCE(SLD.var_pct_pasivo_3m,-9999 ) var_pct_pasivo_3m,
    COALESCE(SLD.var_pct_pasivo_6m,-9999 ) var_pct_pasivo_6m,
    COALESCE(SLD.var_pct_pasivo_12m,-9999 ) var_pct_pasivo_12m,
    COALESCE(SLD.var_pct_colocaciones_1m,-9999 ) var_pct_colocaciones_1m,
    COALESCE(SLD.var_pct_colocaciones_3m,-9999 ) var_pct_colocaciones_3m,
    COALESCE(SLD.var_pct_colocaciones_6m,-9999 ) var_pct_colocaciones_6m,
    COALESCE(SLD.var_pct_colocaciones_12m,-9999 ) var_pct_colocaciones_12m,
    COALESCE(SLD.promedio_movil_pasivo_3m,0 ) promedio_movil_pasivo_3m,
    COALESCE(SLD.promedio_movil_pasivo_6m,0 ) promedio_movil_pasivo_6m,
    COALESCE(SLD.promedio_movil_pasivo_12m,0 ) promedio_movil_pasivo_12m,
    COALESCE(SLD.promedio_movil_colocaciones_3m,0 ) promedio_movil_colocaciones_3m,
    COALESCE(SLD.promedio_movil_colocaciones_6m,0 ) promedio_movil_colocaciones_6m,
    COALESCE(SLD.promedio_movil_colocaciones_12m,0 ) promedio_movil_colocaciones_12m,
    COALESCE(SLD.recencia_pasivo_meses,24 ) recencia_pasivo_meses,
    COALESCE(SLD.recencia_colocaciones_meses,24 ) recencia_colocaciones_meses,
    COALESCE(SLD.recencia_pasivo_mayor_10k_meses,24 ) recencia_pasivo_mayor_10k_meses,
    COALESCE(SLD.recencia_colocaciones_mayor_10k_meses,24 ) recencia_colocaciones_mayor_10k_meses,
    COALESCE(SLD.frecuencia_pasivo_pct_6m,0 ) frecuencia_pasivo_pct_6m,
    COALESCE(SLD.frecuencia_colocaciones_pct_6m,0 ) frecuencia_colocaciones_pct_6m,
    COALESCE(SLD.frecuencia_pasivo_pct_12m,0 ) frecuencia_pasivo_pct_12m,
    COALESCE(SLD.frecuencia_colocaciones_pct_12m,0 ) frecuencia_colocaciones_pct_12m,
    COALESCE(SLD.frecuencia_pasivo_mayor_10k_pct_6m,0 ) frecuencia_pasivo_mayor_10k_pct_6m,
    COALESCE(SLD.frecuencia_colocaciones_mayor_10k_pct_6m,0 ) frecuencia_colocaciones_mayor_10k_pct_6m,
    COALESCE(SLD.frecuencia_crecimiento_pasivo_6m,0 ) frecuencia_crecimiento_pasivo_6m,
    COALESCE(SLD.frecuencia_crecimiento_colocaciones_6m,0 ) frecuencia_crecimiento_colocaciones_6m,
    COALESCE(SLD.meses_con_saldo_pasivo_6m,0 ) meses_con_saldo_pasivo_6m,
    COALESCE(SLD.meses_con_saldo_pasivo_12m,0 ) meses_con_saldo_pasivo_12m,
    COALESCE(SLD.meses_con_saldo_colocaciones_6m,0 ) meses_con_saldo_colocaciones_6m,
    COALESCE(SLD.meses_con_saldo_colocaciones_12m,0 ) meses_con_saldo_colocaciones_12m,
    COALESCE(SLD.flag_crecimiento_pasivo_sostenido_3m,0 ) flag_crecimiento_pasivo_sostenido_3m,
    COALESCE(SLD.flag_crecimiento_colocaciones_sostenido_3m,0 ) flag_crecimiento_colocaciones_sostenido_3m,
    COALESCE(SLD.flag_decrecimiento_pasivo_sostenido_3m,0 ) flag_decrecimiento_pasivo_sostenido_3m,
    COALESCE(SLD.flag_decrecimiento_colocaciones_sostenido_3m,0 ) flag_decrecimiento_colocaciones_sostenido_3m,
    COALESCE(SLD.flg_pasivo_crece_1m,0 ) flg_pasivo_crece_1m,
    COALESCE(SLD.flg_pasivo_decrece_1m,0 ) flg_pasivo_decrece_1m,
    COALESCE(SLD.flg_pasivo_crece_3m,0 ) flg_pasivo_crece_3m,
    COALESCE(SLD.flg_pasivo_decrece_3m,0 ) flg_pasivo_decrece_3m,
    COALESCE(SLD.flg_pasivo_crece_6m,0 ) flg_pasivo_crece_6m,
    COALESCE(SLD.flg_pasivo_decrece_6m,0 ) flg_pasivo_decrece_6m,
    COALESCE(SLD.flg_pasivo_crece_12m,0 ) flg_pasivo_crece_12m,
    COALESCE(SLD.flg_pasivo_decrece_12m,0 ) flg_pasivo_decrece_12m,
    COALESCE(SLD.flg_colocaciones_crece_1m,0 ) flg_colocaciones_crece_1m,
    COALESCE(SLD.flg_colocaciones_decrece_1m,0 ) flg_colocaciones_decrece_1m,
    COALESCE(SLD.flg_colocaciones_crece_3m,0 ) flg_colocaciones_crece_3m,
    COALESCE(SLD.flg_colocaciones_decrece_3m,0 ) flg_colocaciones_decrece_3m,
    COALESCE(SLD.flg_colocaciones_crece_6m,0 ) flg_colocaciones_crece_6m,
    COALESCE(SLD.flg_colocaciones_decrece_6m,0 ) flg_colocaciones_decrece_6m,
    COALESCE(SLD.flg_colocaciones_crece_12m,0 ) flg_colocaciones_crece_12m,
    COALESCE(SLD.flg_colocaciones_decrece_12m,0 ) flg_colocaciones_decrece_12m,
    COALESCE(SLD.flg_pasivo_mayor_1k,0 ) flg_pasivo_mayor_1k,
    COALESCE(SLD.flg_pasivo_mayor_10k,0 ) flg_pasivo_mayor_10k,
    COALESCE(SLD.flg_pasivo_mayor_50k,0 ) flg_pasivo_mayor_50k,
    COALESCE(SLD.flg_pasivo_mayor_100k,0 ) flg_pasivo_mayor_100k,
    COALESCE(SLD.flg_pasivo_mayor_500k,0 ) flg_pasivo_mayor_500k,
    COALESCE(SLD.flg_colocaciones_mayor_1k,0 ) flg_colocaciones_mayor_1k,
    COALESCE(SLD.flg_colocaciones_mayor_10k,0 ) flg_colocaciones_mayor_10k,
    COALESCE(SLD.flg_colocaciones_mayor_50k,0 ) flg_colocaciones_mayor_50k,
    COALESCE(SLD.flg_colocaciones_mayor_100k,0 ) flg_colocaciones_mayor_100k,
    COALESCE(SLD.flg_colocaciones_mayor_500k,0 ) flg_colocaciones_mayor_500k,
    COALESCE(SLD.flg_saldo_total_mayor_1k,0 ) flg_saldo_total_mayor_1k,
    COALESCE(SLD.flg_saldo_total_mayor_10k,0 ) flg_saldo_total_mayor_10k,
    COALESCE(SLD.flg_saldo_total_mayor_50k,0 ) flg_saldo_total_mayor_50k,
    COALESCE(SLD.flg_saldo_total_mayor_100k,0 ) flg_saldo_total_mayor_100k,
    COALESCE(SLD.flg_saldo_total_mayor_500k,0 ) flg_saldo_total_mayor_500k,
    COALESCE(SLD.max_saldo_pasivo_12m,0 ) max_saldo_pasivo_12m,
    COALESCE(SLD.max_saldo_colocaciones_12m,0 ) max_saldo_colocaciones_12m,
    COALESCE(SLD.min_saldo_pasivo_12m,0 ) min_saldo_pasivo_12m,
    COALESCE(SLD.min_saldo_colocaciones_12m,0 ) min_saldo_colocaciones_12m,

    
     --SALDOS AJUSTADOS RCC  EMPRESAS

    COALESCE(SF.deuda_total_max_12m,0) deuda_total_max_12m,    -- DEUDA TOTAL MAXIMA DE EMPRESAS EN EL ULTIMO AÑO
    COALESCE(SF.deuda_total_max_6m,0) deuda_total_max_6m,     -- DEUDA TOTAL MAXIMA DE EMPRESAS EN EL ULTIMOS 6 MESES
    COALESCE(SF.deuda_total_max_3m,0) deuda_total_max_3m,     -- DEUDA TOTAL MAXIMA DE EMPRESAS EN EL ULTIMOS 3 MESES
    COALESCE(SF.deuda_total_min_12m,0) deuda_total_min_12m,    -- DEUDA TOTAL MINIMO DE EMPRESAS EN EL ULTIMOS 3 MESES
    COALESCE(SF.deuda_total_promedio_12m,0) deuda_total_promedio_12m,     -- DEUDA TOTAL PROMEDIO DE EMPRESAS EN EL ULTIMO AÑO
    COALESCE(SF.deuda_total_promedio_6m,0) deuda_total_promedio_6m,        -- DEUDA TOTAL PROMEDIO DE EMPRESAS EN EL ULTIMOS 6 MESES
    COALESCE(SF.deuda_total_promedio_3m,0) deuda_total_promedio_3m,         -- DEUDA TOTAL PROMEDIO DE EMPRESAS EN EL ULTIMOS 3 MESES
    COALESCE(SF.deuda_total_var_pct_12m,-9999999999) deuda_total_var_pct_12m,
    COALESCE(SF.deuda_total_var_pct_6m,-9999999999) deuda_total_var_pct_6m,
    COALESCE(SF.deuda_total_var_pct_3m,-9999999999) deuda_total_var_pct_3m,
    COALESCE(SF.deuda_total_var_pct_1m,-9999999999) deuda_total_var_pct_1m,
    COALESCE(SF.deuda_total_meses_crecimiento_12m,0) deuda_total_meses_crecimiento_12m,
    COALESCE(SF.deuda_total_meses_presencia_12m,0) deuda_total_meses_presencia_12m,
    COALESCE(SF.deuda_total_ratio_actual_vs_historico,0) deuda_total_ratio_actual_vs_historico,
    COALESCE(SF.deuda_ibk_max_12m,0) deuda_ibk_max_12m,
    COALESCE(SF.deuda_ibk_promedio_12m,0) deuda_ibk_promedio_12m,
    COALESCE(SF.deuda_ibk_promedio_6m,0) deuda_ibk_promedio_6m,
    COALESCE(SF.deuda_ibk_var_pct_12m,-9999999999) deuda_ibk_var_pct_12m,
    COALESCE(SF.deuda_ibk_var_pct_6m,-9999999999) deuda_ibk_var_pct_6m,
    COALESCE(SF.deuda_ibk_meses_presencia_12m,0) deuda_ibk_meses_presencia_12m,
    COALESCE(SF.deuda_noibk_max_12m,0) deuda_noibk_max_12m,
    COALESCE(SF.deuda_noibk_promedio_12m,0) deuda_noibk_promedio_12m,
    COALESCE(SF.deuda_noibk_var_pct_12m,-9999999999) deuda_noibk_var_pct_12m,
    COALESCE(SF.sow_ibk_promedio_12m,0) sow_ibk_promedio_12m,
    COALESCE(SF.sow_ibk_promedio_6m,0) sow_ibk_promedio_6m,
    COALESCE(SF.sow_ibk_promedio_3m,0) sow_ibk_promedio_3m,
    COALESCE(SF.sow_ibk_promedio_3m,0) sow_ibk_promedio_3m,
    COALESCE(SF.sow_ibk_max_12m,0) sow_ibk_max_12m,
    COALESCE(SF.sow_ibk_meses_mayor_50_pct,0) sow_ibk_meses_mayor_50_pct,
    COALESCE(SF.cant_empresas_promedio_12m,0) cant_empresas_promedio_12m,
    COALESCE(SF.cant_empresas_max_12m,0) cant_empresas_max_12m,
    COALESCE(SF.ratio_concentracion_bancaria_m1,0) ratio_concentracion_bancaria_m1,


    
    --RCC    


    COALESCE(RCC1.max_atraso_directas_otros,-1) max_atraso_directas_otros,
    COALESCE(RCC1.max_atraso_prestamos_ajustados,-1) max_atraso_prestamos_ajustados,
    COALESCE(RCC1.max_atraso_prestamos_general,-1) max_atraso_prestamos_general,
    COALESCE(RCC1.max_atraso_tar_credito,-1) max_atraso_tar_credito,
    COALESCE(RCC1.max_atraso_impulso,-1) max_atraso_impulso,
    COALESCE(RCC1.max_atraso_reactiva,-1) max_atraso_reactiva,
    COALESCE(RCC1.max_atraso_reprog_fae,-1) max_atraso_reprog_fae,
    COALESCE(RCC1.max_atraso_reprog_emergencia,-1) max_atraso_reprog_emergencia,
    COALESCE(RCC1.max_atraso_coloc_directas_ajustado,-1) max_atraso_coloc_directas_ajustado,
    COALESCE(RCC1.max_atraso_coloc_directas_general,-1) max_atraso_coloc_directas_general,
    COALESCE(RCC1.max_atraso_inmobilario,-1) max_atraso_inmobilario,
    COALESCE(RCC1.max_atraso_tipo_producto_impulso,-1) max_atraso_tipo_producto_impulso,
    COALESCE(RCC1.max_atraso_tipo_producto_reactiva,-1) max_atraso_tipo_producto_reactiva,
    COALESCE(RCC1.max_atraso_tipo_producto_reprog,-1) max_atraso_tipo_producto_reprog,
    COALESCE(RCC1.max_atraso_bco_sobregiros,-1) max_atraso_bco_sobregiros,
    COALESCE(RCC1.max_atraso_bco_directas_otros,-1) max_atraso_bco_directas_otros,
    COALESCE(RCC1.max_atraso_bco_prestamos_ajustados,-1) max_atraso_bco_prestamos_ajustados,
    COALESCE(RCC1.max_atraso_bco_prestamos_general,-1) max_atraso_bco_prestamos_general,
    COALESCE(RCC1.max_atraso_bco_fin_bienes,-1) max_atraso_bco_fin_bienes,
    COALESCE(RCC1.max_atraso_bco_impulso,-1) max_atraso_bco_impulso,
    COALESCE(RCC1.max_atraso_bco_reactiva,-1) max_atraso_bco_reactiva,
    COALESCE(RCC1.max_atraso_bco_reprog_fae,-1) max_atraso_bco_reprog_fae,
    COALESCE(RCC1.max_atraso_bco_reprog_emergencia,-1) max_atraso_bco_reprog_emergencia,
    COALESCE(RCC1.max_atraso_bco_coloc_directas_ajustado,-1) max_atraso_bco_coloc_directas_ajustado,
    COALESCE(RCC1.max_atraso_bco_coloc_directas_general,-1) max_atraso_bco_coloc_directas_general,
    COALESCE(RCC1.max_atraso_bco_inmobilario,-1) max_atraso_bco_inmobilario,
    COALESCE(RCC1.max_atraso_bco_tipo_producto_impulso,-1) max_atraso_bco_tipo_producto_impulso,
    COALESCE(RCC1.max_atraso_bco_tipo_producto_reactiva,-1) max_atraso_bco_tipo_producto_reactiva,
    COALESCE(RCC1.max_atraso_bco_tipo_producto_reprog,-1) max_atraso_bco_tipo_producto_reprog,
    COALESCE(RCC1.max_atraso_ibk_sobregiros,-1) max_atraso_ibk_sobregiros,
    COALESCE(RCC1.max_atraso_ibk_directas_otros,-1) max_atraso_ibk_directas_otros,
    COALESCE(RCC1.max_atraso_ibk_prestamos_ajustados,-1) max_atraso_ibk_prestamos_ajustados,
    COALESCE(RCC1.max_atraso_ibk_prestamos_general,-1) max_atraso_ibk_prestamos_general,
    COALESCE(RCC1.max_atraso_ibk_leasing,-1) max_atraso_ibk_leasing,
    COALESCE(RCC1.max_atraso_ibk_impulso,-1) max_atraso_ibk_impulso,
    COALESCE(RCC1.max_atraso_ibk_reactiva,-1) max_atraso_ibk_reactiva,
    COALESCE(RCC1.max_atraso_ibk_coloc_directas_ajustado,-1) max_atraso_ibk_coloc_directas_ajustado,
    COALESCE(RCC1.max_atraso_ibk_coloc_directas_general,-1) max_atraso_ibk_coloc_directas_general,
    COALESCE(RCC1.max_atraso_ibk_inmobilario,-1) max_atraso_ibk_inmobilario,
    COALESCE(RCC1.max_atraso_ibk_tipo_producto_impulso,-1) max_atraso_ibk_tipo_producto_impulso,
    COALESCE(RCC1.max_atraso_ibk_tipo_producto_reactiva,-1) max_atraso_ibk_tipo_producto_reactiva,
    COALESCE(RCC1.max_atraso_ibk_tipo_producto_reprog,-1) max_atraso_ibk_tipo_producto_reprog,
    COALESCE(RCC1.max_atraso_ibk_tipo_producto_fae,-1) max_atraso_ibk_tipo_producto_fae,
    COALESCE(RCC1.max_atraso_ibk_tipo_producto_pae,-1) max_atraso_ibk_tipo_producto_pae,
    COALESCE(RCC1.max_atraso_competencia_sobregiros,-1) max_atraso_competencia_sobregiros,
    COALESCE(RCC1.max_atraso_competencia_directas_otros,-1) max_atraso_competencia_directas_otros,
    COALESCE(RCC1.max_atraso_competencia_prestamos_ajustados,-1) max_atraso_competencia_prestamos_ajustados,
    COALESCE(RCC1.max_atraso_competencia_prestamos_general,-1) max_atraso_competencia_prestamos_general,
    COALESCE(RCC1.max_atraso_competencia_impulso,-1) max_atraso_competencia_impulso,
    COALESCE(RCC1.max_atraso_competencia_reactiva,-1) max_atraso_competencia_reactiva,
    COALESCE(RCC1.max_atraso_competencia_coloc_directas_ajustado,-1) max_atraso_competencia_coloc_directas_ajustado,
    COALESCE(RCC1.max_atraso_competencia_coloc_directas_general,-1) max_atraso_competencia_coloc_directas_general,
    COALESCE(RCC1.max_atraso_competencia_inmobilario,-1) max_atraso_competencia_inmobilario,
    COALESCE(RCC1.max_atraso_competencia_tipo_producto_impulso,-1) max_atraso_competencia_tipo_producto_impulso,
    COALESCE(RCC1.max_atraso_competencia_tipo_producto_reactiva,-1) max_atraso_competencia_tipo_producto_reactiva,

    COALESCE(RCC2.avg_nro_entidades_cajas_u12m,0) avg_nro_entidades_cajas_u12m,
    COALESCE(RCC2.avg_nro_entidades_financieras_u12m,0) avg_nro_entidades_financieras_u12m,
    COALESCE(RCC2.avg_nro_entidades_bancos_u12m,0) avg_nro_entidades_bancos_u12m,
    COALESCE(RCC2.avg_nro_tipos_entidades_u12m,0) avg_nro_tipos_entidades_u12m,
    COALESCE(RCC2.avg_nro_tipos_productos_u12m,0) avg_nro_tipos_productos_u12m,
    COALESCE(RCC2.avg_nro_productos_u12m,0) avg_nro_productos_u12m,
    COALESCE(RCC2.avg_nro_coloc_directas_normal_u12m,0) avg_nro_coloc_directas_normal_u12m,
    COALESCE(RCC2.avg_nro_reactiva_normal_u12m,0) avg_nro_reactiva_normal_u12m,
    COALESCE(RCC2.avg_nro_impulso_normal_u12m,0) avg_nro_impulso_normal_u12m,
    COALESCE(RCC2.avg_nro_reprogramado_normal_u12m,0) avg_nro_reprogramado_normal_u12m,
    COALESCE(RCC2.avg_nro_reg_coloc_directas_bancos_ajustado_u12m,0) avg_nro_reg_coloc_directas_bancos_ajustado_u12m,
    COALESCE(RCC2.avg_nro_reg_coloc_directas_bancos_general_u12m,0) avg_nro_reg_coloc_directas_bancos_general_u12m,
    COALESCE(RCC2.avg_nro_reg_coloc_indirectas_bancos_u12m,0) avg_nro_reg_coloc_indirectas_bancos_u12m,
    COALESCE(RCC2.avg_nro_reg_garantias_bancos_u12m,0) avg_nro_reg_garantias_bancos_u12m,
    COALESCE(RCC2.avg_nro_reg_reprogramados_bancos_u12m,0) avg_nro_reg_reprogramados_bancos_u12m,
    COALESCE(RCC2.avg_nro_reg_coloc_directas_cajas_financ_ajustado_u12m,0) avg_nro_reg_coloc_directas_cajas_financ_ajustado_u12m,
    COALESCE(RCC2.avg_nro_reg_coloc_directas_cajas_financ_general_u12m,0) avg_nro_reg_coloc_directas_cajas_financ_general_u12m,
    COALESCE(RCC2.avg_nro_reg_coloc_indirectas_cajas_financ_u12m,0) avg_nro_reg_coloc_indirectas_cajas_financ_u12m,
    COALESCE(RCC2.avg_nro_reg_garantias_cajas_financ_u12m,0) avg_nro_reg_garantias_cajas_financ_u12m,
    COALESCE(RCC2.avg_nro_reg_reprogramados_cajas_financ_u12m,0) avg_nro_reg_reprogramados_cajas_financ_u12m,
    COALESCE(RCC2.avg_saldo_castigado_amt_u12m,0) avg_saldo_castigado_amt_u12m,
    COALESCE(RCC2.avg_saldo_coloc_directas_amt_general_u12m,0) avg_saldo_coloc_directas_amt_general_u12m,
    COALESCE(RCC2.avg_saldo_coloc_directas_ajustado_amt_u12m,0) avg_saldo_coloc_directas_ajustado_amt_u12m,
    COALESCE(RCC2.avg_saldo_coloc_indirectas_amt_u12m,0) avg_saldo_coloc_indirectas_amt_u12m,
    COALESCE(RCC2.avg_saldo_garantias_amt_u12m,0) avg_saldo_garantias_amt_u12m,
    COALESCE(RCC2.avg_saldo_reprogramados_amt_u12m,0) avg_saldo_reprogramados_amt_u12m,
    COALESCE(RCC2.avg_saldo_coloc_directas_general_vig_amt_u12m,0) avg_saldo_coloc_directas_general_vig_amt_u12m,
    COALESCE(RCC2.avg_saldo_coloc_directas_vig_amt_u12m,0) avg_saldo_coloc_directas_vig_amt_u12m,
    COALESCE(RCC2.avg_saldo_reprogramados_vig_amt_u12m,0) avg_saldo_reprogramados_vig_amt_u12m,
    COALESCE(RCC2.avg_saldo_coloc_directas_general_no_vig_amt_u12m,0) avg_saldo_coloc_directas_general_no_vig_amt_u12m,
    COALESCE(RCC2.avg_saldo_coloc_directas_no_vig_amt_u12m,0) avg_saldo_coloc_directas_no_vig_amt_u12m,
    COALESCE(RCC2.avg_saldo_reprogramados_no_vig_amt_u12m,0) avg_saldo_reprogramados_no_vig_amt_u12m,
    COALESCE(RCC2.avg_saldo_garantias_otros_amt_u12m,0) avg_saldo_garantias_otros_amt_u12m,
    COALESCE(RCC2.avg_saldo_garantias_no_pref_amt_u12m,0) avg_saldo_garantias_no_pref_amt_u12m,
    COALESCE(RCC2.avg_saldo_garantias_hipo_amt_u12m,0) avg_saldo_garantias_hipo_amt_u12m,
    COALESCE(RCC2.avg_saldo_garantias_auto_amt_u12m,0) avg_saldo_garantias_auto_amt_u12m,
    COALESCE(RCC2.avg_saldo_garantias_warrants_amt_u12m,0) avg_saldo_garantias_warrants_amt_u12m,
    COALESCE(RCC2.avg_saldo_garantias_pref_amt_u12m,0) avg_saldo_garantias_pref_amt_u12m,
    COALESCE(RCC2.avg_saldo_bco_castigado_amt_u12m,0) avg_saldo_bco_castigado_amt_u12m,
    COALESCE(RCC2.avg_saldo_bco_coloc_directas_amt_general_u12m,0) avg_saldo_bco_coloc_directas_amt_general_u12m,
    COALESCE(RCC2.avg_saldo_bco_coloc_directas_ajustado_amt_u12m,0) avg_saldo_bco_coloc_directas_ajustado_amt_u12m,
    COALESCE(RCC2.avg_saldo_bco_coloc_indirectas_amt_u12m,0) avg_saldo_bco_coloc_indirectas_amt_u12m,
    COALESCE(RCC2.avg_saldo_bco_garantias_amt_u12m,0) avg_saldo_bco_garantias_amt_u12m,
    COALESCE(RCC2.avg_saldo_bco_reprogramados_amt_u12m,0) avg_saldo_bco_reprogramados_amt_u12m,
    COALESCE(RCC2.avg_saldo_bco_coloc_directas_general_vig_amt_u12m,0) avg_saldo_bco_coloc_directas_general_vig_amt_u12m,
    COALESCE(RCC2.avg_saldo_bco_coloc_directas_vig_amt_u12m,0) avg_saldo_bco_coloc_directas_vig_amt_u12m,
    COALESCE(RCC2.avg_saldo_bco_reprogramados_vig_amt_u12m,0) avg_saldo_bco_reprogramados_vig_amt_u12m,
    COALESCE(RCC2.avg_saldo_bco_coloc_directas_general_no_vig_amt_u12m,0) avg_saldo_bco_coloc_directas_general_no_vig_amt_u12m,
    COALESCE(RCC2.avg_saldo_bco_coloc_directas_no_vig_amt_u12m,0) avg_saldo_bco_coloc_directas_no_vig_amt_u12m,
    COALESCE(RCC2.avg_saldo_bco_reprogramados_no_vig_amt_u12m,0) avg_saldo_bco_reprogramados_no_vig_amt_u12m,
    COALESCE(RCC2.avg_saldo_bco_garantias_otros_amt_u12m,0) avg_saldo_bco_garantias_otros_amt_u12m,
    COALESCE(RCC2.avg_saldo_bco_garantias_no_pref_amt_u12m,0) avg_saldo_bco_garantias_no_pref_amt_u12m,
    COALESCE(RCC2.avg_saldo_bco_garantias_hipo_amt_u12m,0) avg_saldo_bco_garantias_hipo_amt_u12m,
    COALESCE(RCC2.avg_saldo_bco_garantias_auto_amt_u12m,0) avg_saldo_bco_garantias_auto_amt_u12m,
    COALESCE(RCC2.avg_saldo_bco_garantias_warrants_amt_u12m,0) avg_saldo_bco_garantias_warrants_amt_u12m,
    COALESCE(RCC2.avg_saldo_bco_garantias_pref_amt_u12m,0) avg_saldo_bco_garantias_pref_amt_u12m,
    COALESCE(RCC2.avg_saldo_ibk_castigado_amt_u12m,0) avg_saldo_ibk_castigado_amt_u12m,
    COALESCE(RCC2.avg_saldo_ibk_coloc_directas_amt_general_u12m,0) avg_saldo_ibk_coloc_directas_amt_general_u12m,
    COALESCE(RCC2.avg_saldo_ibk_coloc_directas_ajustado_amt_u12m,0) avg_saldo_ibk_coloc_directas_ajustado_amt_u12m,
    COALESCE(RCC2.avg_saldo_ibk_coloc_indirectas_amt_u12m,0) avg_saldo_ibk_coloc_indirectas_amt_u12m,
    COALESCE(RCC2.avg_saldo_ibk_garantias_amt_u12m,0) avg_saldo_ibk_garantias_amt_u12m,
    COALESCE(RCC2.avg_saldo_ibk_reprogramados_amt_u12m,0) avg_saldo_ibk_reprogramados_amt_u12m,
    COALESCE(RCC2.avg_saldo_ibk_coloc_directas_general_vig_amt_u12m,0) avg_saldo_ibk_coloc_directas_general_vig_amt_u12m,
    COALESCE(RCC2.avg_saldo_ibk_coloc_directas_vig_amt_u12m,0) avg_saldo_ibk_coloc_directas_vig_amt_u12m,
    COALESCE(RCC2.avg_saldo_ibk_reprogramados_vig_amt_u12m,0) avg_saldo_ibk_reprogramados_vig_amt_u12m,
    COALESCE(RCC2.avg_saldo_ibk_coloc_directas_general_no_vig_amt_u12m,0) avg_saldo_ibk_coloc_directas_general_no_vig_amt_u12m,
    COALESCE(RCC2.avg_saldo_ibk_coloc_directas_no_vig_amt_u12m,0) avg_saldo_ibk_coloc_directas_no_vig_amt_u12m,
    COALESCE(RCC2.avg_saldo_ibk_reprogramados_no_vig_amt_u12m,0) avg_saldo_ibk_reprogramados_no_vig_amt_u12m,
    COALESCE(RCC2.avg_saldo_ibk_garantias_otros_amt_u12m,0) avg_saldo_ibk_garantias_otros_amt_u12m,
    COALESCE(RCC2.avg_saldo_ibk_garantias_no_pref_amt_u12m,0) avg_saldo_ibk_garantias_no_pref_amt_u12m,
    COALESCE(RCC2.avg_saldo_ibk_garantias_hipo_amt_u12m,0) avg_saldo_ibk_garantias_hipo_amt_u12m,
    COALESCE(RCC2.avg_saldo_ibk_garantias_auto_amt_u12m,0) avg_saldo_ibk_garantias_auto_amt_u12m,
    COALESCE(RCC2.avg_saldo_ibk_garantias_warrants_amt_u12m,0) avg_saldo_ibk_garantias_warrants_amt_u12m,
    COALESCE(RCC2.avg_saldo_ibk_garantias_pref_amt_u12m,0) avg_saldo_ibk_garantias_pref_amt_u12m,
    COALESCE(RCC2.avg_saldo_bco_no_ibk_castigado_amt_u12m,0) avg_saldo_bco_no_ibk_castigado_amt_u12m,
    COALESCE(RCC2.avg_saldo_bco_no_ibk_coloc_directas_amt_general_u12m,0) avg_saldo_bco_no_ibk_coloc_directas_amt_general_u12m,
    COALESCE(RCC2.avg_saldo_bco_no_ibk_coloc_directas_ajustado_amt_u12m,0) avg_saldo_bco_no_ibk_coloc_directas_ajustado_amt_u12m,
    COALESCE(RCC2.avg_saldo_bco_no_ibk_coloc_indirectas_amt_u12m,0) avg_saldo_bco_no_ibk_coloc_indirectas_amt_u12m,
    COALESCE(RCC2.avg_saldo_bco_no_ibk_garantias_amt_u12m,0) avg_saldo_bco_no_ibk_garantias_amt_u12m,
    COALESCE(RCC2.avg_saldo_bco_no_ibk_reprogramados_amt_u12m,0) avg_saldo_bco_no_ibk_reprogramados_amt_u12m,
    COALESCE(RCC2.avg_saldo_bco_no_ibk_coloc_directas_general_vig_amt_u12m,0) avg_saldo_bco_no_ibk_coloc_directas_general_vig_amt_u12m,
    COALESCE(RCC2.avg_saldo_bco_no_ibk_coloc_directas_vig_amt_u12m,0) avg_saldo_bco_no_ibk_coloc_directas_vig_amt_u12m,
    COALESCE(RCC2.avg_saldo_bco_no_ibk_reprogramados_vig_amt_u12m,0) avg_saldo_bco_no_ibk_reprogramados_vig_amt_u12m,
    COALESCE(RCC2.avg_saldo_bco_no_ibk_coloc_directas_general_no_vig_amt_u12m,0) avg_saldo_bco_no_ibk_coloc_directas_general_no_vig_amt_u12m,
    COALESCE(RCC2.avg_saldo_bco_no_ibk_coloc_directas_no_vig_amt_u12m,0) avg_saldo_bco_no_ibk_coloc_directas_no_vig_amt_u12m,
    COALESCE(RCC2.avg_saldo_bco_no_ibk_reprogramados_no_vig_amt_u12m,0) avg_saldo_bco_no_ibk_reprogramados_no_vig_amt_u12m,
    COALESCE(RCC2.avg_saldo_bco_no_ibk_garantias_otros_amt_u12m,0) avg_saldo_bco_no_ibk_garantias_otros_amt_u12m,
    COALESCE(RCC2.avg_saldo_bco_no_ibk_garantias_no_pref_amt_u12m,0) avg_saldo_bco_no_ibk_garantias_no_pref_amt_u12m,
    COALESCE(RCC2.avg_saldo_bco_no_ibk_garantias_hipo_amt_u12m,0) avg_saldo_bco_no_ibk_garantias_hipo_amt_u12m,
    COALESCE(RCC2.avg_saldo_bco_no_ibk_garantias_auto_amt_u12m,0) avg_saldo_bco_no_ibk_garantias_auto_amt_u12m,
    COALESCE(RCC2.avg_saldo_bco_no_ibk_garantias_warrants_amt_u12m,0) avg_saldo_bco_no_ibk_garantias_warrants_amt_u12m,
    COALESCE(RCC2.avg_saldo_bco_no_ibk_garantias_pref_amt_u12m,0) avg_saldo_bco_no_ibk_garantias_pref_amt_u12m,
    COALESCE(RCC2.avg_saldo_cajas_financ_castigado_amt_u12m,0) avg_saldo_cajas_financ_castigado_amt_u12m,
    COALESCE(RCC2.avg_saldo_cajas_financ_coloc_directas_amt_general_u12m,0) avg_saldo_cajas_financ_coloc_directas_amt_general_u12m,
    COALESCE(RCC2.avg_saldo_cajas_financ_coloc_directas_ajustado_amt_u12m,0) avg_saldo_cajas_financ_coloc_directas_ajustado_amt_u12m,
    COALESCE(RCC2.avg_saldo_cajas_financ_coloc_indirectas_amt_u12m,0) avg_saldo_cajas_financ_coloc_indirectas_amt_u12m,
    COALESCE(RCC2.avg_saldo_cajas_financ_garantias_amt_u12m,0) avg_saldo_cajas_financ_garantias_amt_u12m,
    COALESCE(RCC2.avg_saldo_cajas_financ_reprogramados_amt_u12m,0) avg_saldo_cajas_financ_reprogramados_amt_u12m,
    COALESCE(RCC2.avg_saldo_cajas_financ_coloc_directas_general_vig_amt_u12m,0) avg_saldo_cajas_financ_coloc_directas_general_vig_amt_u12m,
    COALESCE(RCC2.avg_saldo_cajas_financ_coloc_directas_vig_amt_u12m,0) avg_saldo_cajas_financ_coloc_directas_vig_amt_u12m,
    COALESCE(RCC2.avg_saldo_cajas_financ_reprogramados_vig_amt_u12m,0) avg_saldo_cajas_financ_reprogramados_vig_amt_u12m,
    COALESCE(RCC2.avg_saldo_cajas_financ_coloc_directas_general_no_vig_amt_u12m,0) avg_saldo_cajas_financ_coloc_directas_general_no_vig_amt_u12m,
    COALESCE(RCC2.avg_saldo_cajas_financ_coloc_directas_no_vig_amt_u12m,0) avg_saldo_cajas_financ_coloc_directas_no_vig_amt_u12m,
    COALESCE(RCC2.avg_saldo_cajas_financ_reprogramados_no_vig_amt_u12m,0) avg_saldo_cajas_financ_reprogramados_no_vig_amt_u12m,
    COALESCE(RCC2.avg_saldo_cajas_financ_garantias_otros_amt_u12m,0) avg_saldo_cajas_financ_garantias_otros_amt_u12m,
    COALESCE(RCC2.avg_saldo_cajas_financ_garantias_no_pref_amt_u12m,0) avg_saldo_cajas_financ_garantias_no_pref_amt_u12m,
    COALESCE(RCC2.avg_saldo_cajas_financ_garantias_hipo_amt_u12m,0) avg_saldo_cajas_financ_garantias_hipo_amt_u12m,
    COALESCE(RCC2.avg_saldo_cajas_financ_garantias_auto_amt_u12m,0) avg_saldo_cajas_financ_garantias_auto_amt_u12m,
    COALESCE(RCC2.avg_saldo_cajas_financ_garantias_warrants_amt_u12m,0) avg_saldo_cajas_financ_garantias_warrants_amt_u12m,
    COALESCE(RCC2.avg_saldo_cajas_financ_garantias_pref_amt_u12m,0) avg_saldo_cajas_financ_garantias_pref_amt_u12m,
    COALESCE(RCC2.avg_saldo_competencia_castigado_amt_u12m,0) avg_saldo_competencia_castigado_amt_u12m,
    COALESCE(RCC2.avg_saldo_competencia_coloc_directas_amt_general_u12m,0) avg_saldo_competencia_coloc_directas_amt_general_u12m,
    COALESCE(RCC2.avg_saldo_competencia_coloc_directas_ajustado_amt_u12m,0) avg_saldo_competencia_coloc_directas_ajustado_amt_u12m,
    COALESCE(RCC2.avg_saldo_competencia_coloc_indirectas_amt_u12m,0) avg_saldo_competencia_coloc_indirectas_amt_u12m,
    COALESCE(RCC2.avg_saldo_competencia_garantias_amt_u12m,0) avg_saldo_competencia_garantias_amt_u12m,
    COALESCE(RCC2.avg_saldo_competencia_reprogramados_amt_u12m,0) avg_saldo_competencia_reprogramados_amt_u12m,
    COALESCE(RCC2.avg_saldo_competencia_coloc_directas_general_vig_amt_u12m,0) avg_saldo_competencia_coloc_directas_general_vig_amt_u12m,
    COALESCE(RCC2.avg_saldo_competencia_coloc_directas_vig_amt_u12m,0) avg_saldo_competencia_coloc_directas_vig_amt_u12m,
    COALESCE(RCC2.avg_saldo_competencia_reprogramados_vig_amt_u12m,0) avg_saldo_competencia_reprogramados_vig_amt_u12m,
    COALESCE(RCC2.avg_saldo_competencia_coloc_directas_general_no_vig_amt_u12m,0) avg_saldo_competencia_coloc_directas_general_no_vig_amt_u12m,
    COALESCE(RCC2.avg_saldo_competencia_coloc_directas_no_vig_amt_u12m,0) avg_saldo_competencia_coloc_directas_no_vig_amt_u12m,
    COALESCE(RCC2.avg_saldo_competencia_reprogramados_no_vig_amt_u12m,0) avg_saldo_competencia_reprogramados_no_vig_amt_u12m,
    COALESCE(RCC2.avg_saldo_competencia_garantias_otros_amt_u12m,0) avg_saldo_competencia_garantias_otros_amt_u12m,
    COALESCE(RCC2.avg_saldo_competencia_garantias_no_pref_amt_u12m,0) avg_saldo_competencia_garantias_no_pref_amt_u12m,
    COALESCE(RCC2.avg_saldo_competencia_garantias_hipo_amt_u12m,0) avg_saldo_competencia_garantias_hipo_amt_u12m,
    COALESCE(RCC2.avg_saldo_competencia_garantias_auto_amt_u12m,0) avg_saldo_competencia_garantias_auto_amt_u12m,
    COALESCE(RCC2.avg_saldo_competencia_garantias_warrants_amt_u12m,0) avg_saldo_competencia_garantias_warrants_amt_u12m,
    COALESCE(RCC2.avg_saldo_competencia_garantias_pref_amt_u12m,0) avg_saldo_competencia_garantias_pref_amt_u12m,
    COALESCE(RCC2.avg_saldo_sobregiros_amt_u12m,0) avg_saldo_sobregiros_amt_u12m,
    COALESCE(RCC2.avg_saldo_comex_amt_u12m,0) avg_saldo_comex_amt_u12m,
    COALESCE(RCC2.avg_saldo_descuentos_amt_u12m,0) avg_saldo_descuentos_amt_u12m,
    COALESCE(RCC2.avg_saldo_directas_otros_amt_u12m,0) avg_saldo_directas_otros_amt_u12m,
    COALESCE(RCC2.avg_saldo_factoring_amt_u12m,0) avg_saldo_factoring_amt_u12m,
    COALESCE(RCC2.avg_saldo_prestamos_ajustados_amt_u12m,0) avg_saldo_prestamos_ajustados_amt_u12m,
    COALESCE(RCC2.avg_saldo_prestamos_general_amt_u12m,0) avg_saldo_prestamos_general_amt_u12m,
    COALESCE(RCC2.avg_saldo_leasing_amt_u12m,0) avg_saldo_leasing_amt_u12m,
    COALESCE(RCC2.avg_saldo_tar_credito_amt_u12m,0) avg_saldo_tar_credito_amt_u12m,
    COALESCE(RCC2.avg_saldo_carta_credito_amt_u12m,0) avg_saldo_carta_credito_amt_u12m,
    COALESCE(RCC2.avg_saldo_carta_fianza_amt_u12m,0) avg_saldo_carta_fianza_amt_u12m,
    COALESCE(RCC2.avg_saldo_deriv_instr_amt_u12m,0) avg_saldo_deriv_instr_amt_u12m,
    COALESCE(RCC2.avg_saldo_deriv_moenda_amt_u12m,0) avg_saldo_deriv_moenda_amt_u12m,
    COALESCE(RCC2.avg_saldo_fin_fae_amt_u12m,0) avg_saldo_fin_fae_amt_u12m,
    COALESCE(RCC2.avg_saldo_gar_fae_amt_u12m,0) avg_saldo_gar_fae_amt_u12m,
    COALESCE(RCC2.avg_saldo_gar_impulso_amt_u12m,0) avg_saldo_gar_impulso_amt_u12m,
    COALESCE(RCC2.avg_saldo_gar_pae_amt_u12m,0) avg_saldo_gar_pae_amt_u12m,
    COALESCE(RCC2.avg_saldo_gar_reactiva_amt_u12m,0) avg_saldo_gar_reactiva_amt_u12m,
    COALESCE(RCC2.avg_saldo_inmuebeles_amt_u12m,0) avg_saldo_inmuebeles_amt_u12m,
    COALESCE(RCC2.avg_saldo_fin_bienes_amt_u12m,0) avg_saldo_fin_bienes_amt_u12m,
    COALESCE(RCC2.avg_saldo_impulso_amt_u12m,0) avg_saldo_impulso_amt_u12m,
    COALESCE(RCC2.avg_saldo_reactiva_amt_u12m,0) avg_saldo_reactiva_amt_u12m,
    COALESCE(RCC2.avg_saldo_reprog_fae_amt_u12m,0) avg_saldo_reprog_fae_amt_u12m,
    COALESCE(RCC2.avg_saldo_reprog_emergencia_amt_u12m,0) avg_saldo_reprog_emergencia_amt_u12m,
    COALESCE(RCC2.avg_saldo_vig_sobregiros_amt_u12m,0) avg_saldo_vig_sobregiros_amt_u12m,
    COALESCE(RCC2.avg_saldo_vig_comex_amt_u12m,0) avg_saldo_vig_comex_amt_u12m,
    COALESCE(RCC2.avg_saldo_vig_descuentos_amt_u12m,0) avg_saldo_vig_descuentos_amt_u12m,
    COALESCE(RCC2.avg_saldo_vig_directas_otros_amt_u12m,0) avg_saldo_vig_directas_otros_amt_u12m,
    COALESCE(RCC2.avg_saldo_vig_factoring_amt_u12m,0) avg_saldo_vig_factoring_amt_u12m,
    COALESCE(RCC2.avg_saldo_vig_prestamos_general_amt_u12m,0) avg_saldo_vig_prestamos_general_amt_u12m,
    COALESCE(RCC2.avg_saldo_vig_leasing_amt_u12m,0) avg_saldo_vig_leasing_amt_u12m,
    COALESCE(RCC2.avg_saldo_vig_tar_credito_amt_u12m,0) avg_saldo_vig_tar_credito_amt_u12m,
    COALESCE(RCC2.avg_saldo_vig_impulso_amt_u12m,0) avg_saldo_vig_impulso_amt_u12m,
    COALESCE(RCC2.avg_saldo_vig_reactiva_amt_u12m,0) avg_saldo_vig_reactiva_amt_u12m,
    COALESCE(RCC2.avg_saldo_vig_reprog_fae_amt_u12m,0) avg_saldo_vig_reprog_fae_amt_u12m,
    COALESCE(RCC2.avg_saldo_vig_reprog_emergencia_amt_u12m,0) avg_saldo_vig_reprog_emergencia_amt_u12m,
    COALESCE(RCC2.avg_saldo_no_vig_sobregiros_amt_u12m,0) avg_saldo_no_vig_sobregiros_amt_u12m,
    COALESCE(RCC2.avg_saldo_no_vig_comex_amt_u12m,0) avg_saldo_no_vig_comex_amt_u12m,
    COALESCE(RCC2.avg_saldo_no_vig_directas_otros_amt_u12m,0) avg_saldo_no_vig_directas_otros_amt_u12m,
    COALESCE(RCC2.avg_saldo_no_vig_factoring_amt_u12m,0) avg_saldo_no_vig_factoring_amt_u12m,
    COALESCE(RCC2.avg_saldo_no_vig_prestamos_general_amt_u12m,0) avg_saldo_no_vig_prestamos_general_amt_u12m,
    COALESCE(RCC2.avg_saldo_no_vig_leasing_amt_u12m,0) avg_saldo_no_vig_leasing_amt_u12m,
    COALESCE(RCC2.avg_saldo_no_vig_tar_credito_amt_u12m,0) avg_saldo_no_vig_tar_credito_amt_u12m,
    COALESCE(RCC2.avg_saldo_no_vig_fin_bienes_amt_u12m,0) avg_saldo_no_vig_fin_bienes_amt_u12m,
    COALESCE(RCC2.avg_saldo_no_vig_impulso_amt_u12m,0) avg_saldo_no_vig_impulso_amt_u12m,
    COALESCE(RCC2.avg_saldo_no_vig_reactiva_amt_u12m,0) avg_saldo_no_vig_reactiva_amt_u12m,
    COALESCE(RCC2.avg_saldo_no_vig_reprog_fae_amt_u12m,0) avg_saldo_no_vig_reprog_fae_amt_u12m,
    COALESCE(RCC2.avg_saldo_no_vig_reprog_emergencia_amt_u12m,0) avg_saldo_no_vig_reprog_emergencia_amt_u12m,
    COALESCE(RCC2.max_atraso_sobregiros_u12m,-1) max_atraso_sobregiros_u12m,
    COALESCE(RCC2.max_atraso_comex_u12m,-1) max_atraso_comex_u12m,
    COALESCE(RCC2.max_atraso_descuentos_u12m,-1) max_atraso_descuentos_u12m,
    COALESCE(RCC2.max_atraso_directas_otros_u12m,-1) max_atraso_directas_otros_u12m,
    COALESCE(RCC2.max_atraso_factoring_u12m,-1) max_atraso_factoring_u12m,
    COALESCE(RCC2.max_atraso_prestamos_ajustados_u12m,-1) max_atraso_prestamos_ajustados_u12m,
    COALESCE(RCC2.max_atraso_prestamos_general_u12m,-1) max_atraso_prestamos_general_u12m,
    COALESCE(RCC2.max_atraso_leasing_u12m,-1) max_atraso_leasing_u12m,
    COALESCE(RCC2.max_atraso_tar_credito_u12m,-1) max_atraso_tar_credito_u12m,
    COALESCE(RCC2.max_atraso_inmuebeles_u12m,-1) max_atraso_inmuebeles_u12m,
    COALESCE(RCC2.max_atraso_fin_bienes_u12m,-1) max_atraso_fin_bienes_u12m,
    COALESCE(RCC2.max_atraso_impulso_u12m,-1) max_atraso_impulso_u12m,
    COALESCE(RCC2.max_atraso_reactiva_u12m,-1) max_atraso_reactiva_u12m,
    COALESCE(RCC2.max_atraso_reprog_fae_u12m,-1) max_atraso_reprog_fae_u12m,
    COALESCE(RCC2.max_atraso_reprog_emergencia_u12m,-1) max_atraso_reprog_emergencia_u12m,
    COALESCE(RCC2.max_atraso_coloc_directas_ajustado_u12m,-1) max_atraso_coloc_directas_ajustado_u12m,
    COALESCE(RCC2.max_atraso_coloc_directas_general_u12m,-1) max_atraso_coloc_directas_general_u12m,
    --COALESCE(RCC2.max_atraso_inmobilario_u12m,-1) max_atraso_inmobilario_u12m,
    COALESCE(RCC2.max_atraso_tipo_producto_impulso_u12m,-1) max_atraso_tipo_producto_impulso_u12m,
    COALESCE(RCC2.max_atraso_tipo_producto_reactiva_u12m,-1) max_atraso_tipo_producto_reactiva_u12m,
    COALESCE(RCC2.max_atraso_tipo_producto_reprog_u12m,-1) max_atraso_tipo_producto_reprog_u12m,
    COALESCE(RCC2.max_atraso_tipo_producto_fae_u12m,-1) max_atraso_tipo_producto_fae_u12m,
    COALESCE(RCC2.max_atraso_tipo_producto_pae_u12m,-1) max_atraso_tipo_producto_pae_u12m,
    COALESCE(RCC2.max_atraso_bco_sobregiros_u12m,-1) max_atraso_bco_sobregiros_u12m,
    COALESCE(RCC2.max_atraso_bco_comex_u12m,-1) max_atraso_bco_comex_u12m,
    COALESCE(RCC2.max_atraso_bco_descuentos_u12m,-1) max_atraso_bco_descuentos_u12m,
    COALESCE(RCC2.max_atraso_bco_directas_otros_u12m,-1) max_atraso_bco_directas_otros_u12m,
    COALESCE(RCC2.max_atraso_bco_factoring_u12m,-1) max_atraso_bco_factoring_u12m,
    COALESCE(RCC2.max_atraso_bco_prestamos_ajustados_u12m,-1) max_atraso_bco_prestamos_ajustados_u12m,
    COALESCE(RCC2.max_atraso_bco_prestamos_general_u12m,-1) max_atraso_bco_prestamos_general_u12m,
    COALESCE(RCC2.max_atraso_bco_leasing_u12m,-1) max_atraso_bco_leasing_u12m,
    COALESCE(RCC2.max_atraso_bco_tar_credito_u12m,-1) max_atraso_bco_tar_credito_u12m,
    COALESCE(RCC2.max_atraso_bco_inmuebeles_u12m,-1) max_atraso_bco_inmuebeles_u12m,
    COALESCE(RCC2.max_atraso_bco_fin_bienes_u12m,-1) max_atraso_bco_fin_bienes_u12m,
    COALESCE(RCC2.max_atraso_bco_impulso_u12m,-1) max_atraso_bco_impulso_u12m,
    COALESCE(RCC2.max_atraso_bco_reactiva_u12m,-1) max_atraso_bco_reactiva_u12m,
    COALESCE(RCC2.max_atraso_bco_reprog_fae_u12m,-1) max_atraso_bco_reprog_fae_u12m,
    COALESCE(RCC2.max_atraso_bco_reprog_emergencia_u12m,-1) max_atraso_bco_reprog_emergencia_u12m,
    COALESCE(RCC2.max_atraso_bco_coloc_directas_ajustado_u12m,-1) max_atraso_bco_coloc_directas_ajustado_u12m,
    COALESCE(RCC2.max_atraso_bco_coloc_directas_general_u12m,-1) max_atraso_bco_coloc_directas_general_u12m,
    --COALESCE(RCC2.max_atraso_bco_inmobilario_u12m,-1) max_atraso_bco_inmobilario_u12m,
    COALESCE(RCC2.max_atraso_bco_tipo_producto_impulso_u12m,-1) max_atraso_bco_tipo_producto_impulso_u12m,
    COALESCE(RCC2.max_atraso_bco_tipo_producto_reactiva_u12m,-1) max_atraso_bco_tipo_producto_reactiva_u12m,
    COALESCE(RCC2.max_atraso_bco_tipo_producto_reprog_u12m,-1) max_atraso_bco_tipo_producto_reprog_u12m,
    COALESCE(RCC2.max_atraso_bco_tipo_producto_fae_u12m,-1) max_atraso_bco_tipo_producto_fae_u12m,
    COALESCE(RCC2.max_atraso_bco_tipo_producto_pae_u12m,-1) max_atraso_bco_tipo_producto_pae_u12m,
    COALESCE(RCC2.max_atraso_ibk_sobregiros_u12m,-1) max_atraso_ibk_sobregiros_u12m,
    COALESCE(RCC2.max_atraso_ibk_comex_u12m,-1) max_atraso_ibk_comex_u12m,
    COALESCE(RCC2.max_atraso_ibk_descuentos_u12m,-1) max_atraso_ibk_descuentos_u12m,
    COALESCE(RCC2.max_atraso_ibk_directas_otros_u12m,-1) max_atraso_ibk_directas_otros_u12m,
    COALESCE(RCC2.max_atraso_ibk_factoring_u12m,-1) max_atraso_ibk_factoring_u12m,
    COALESCE(RCC2.max_atraso_ibk_prestamos_ajustados_u12m,-1) max_atraso_ibk_prestamos_ajustados_u12m,
    COALESCE(RCC2.max_atraso_ibk_prestamos_general_u12m,-1) max_atraso_ibk_prestamos_general_u12m,
    COALESCE(RCC2.max_atraso_ibk_leasing_u12m,-1) max_atraso_ibk_leasing_u12m,
    COALESCE(RCC2.max_atraso_ibk_tar_credito_u12m,-1) max_atraso_ibk_tar_credito_u12m,
    COALESCE(RCC2.max_atraso_ibk_inmuebeles_u12m,-1) max_atraso_ibk_inmuebeles_u12m,
    COALESCE(RCC2.max_atraso_ibk_fin_bienes_u12m,-1) max_atraso_ibk_fin_bienes_u12m,
    COALESCE(RCC2.max_atraso_ibk_impulso_u12m,-1) max_atraso_ibk_impulso_u12m,
    COALESCE(RCC2.max_atraso_ibk_reactiva_u12m,-1) max_atraso_ibk_reactiva_u12m,
    COALESCE(RCC2.max_atraso_ibk_reprog_fae_u12m,-1) max_atraso_ibk_reprog_fae_u12m,
    COALESCE(RCC2.max_atraso_ibk_reprog_emergencia_u12m,-1) max_atraso_ibk_reprog_emergencia_u12m,
    COALESCE(RCC2.max_atraso_ibk_coloc_directas_ajustado_u12m,-1) max_atraso_ibk_coloc_directas_ajustado_u12m,
    COALESCE(RCC2.max_atraso_ibk_coloc_directas_general_u12m,-1) max_atraso_ibk_coloc_directas_general_u12m,
    --COALESCE(RCC2.max_atraso_ibk_inmobilario_u12m,-1) max_atraso_ibk_inmobilario_u12m,
    COALESCE(RCC2.max_atraso_ibk_tipo_producto_impulso_u12m,-1) max_atraso_ibk_tipo_producto_impulso_u12m,
    COALESCE(RCC2.max_atraso_ibk_tipo_producto_reactiva_u12m,-1) max_atraso_ibk_tipo_producto_reactiva_u12m,
    COALESCE(RCC2.max_atraso_ibk_tipo_producto_reprog_u12m,-1) max_atraso_ibk_tipo_producto_reprog_u12m,
    COALESCE(RCC2.max_atraso_ibk_tipo_producto_fae_u12m,-1) max_atraso_ibk_tipo_producto_fae_u12m,
    COALESCE(RCC2.max_atraso_ibk_tipo_producto_pae_u12m,-1) max_atraso_ibk_tipo_producto_pae_u12m,
    COALESCE(RCC2.max_atraso_competencia_sobregiros_u12m,-1) max_atraso_competencia_sobregiros_u12m,
    COALESCE(RCC2.max_atraso_competencia_comex_u12m,-1) max_atraso_competencia_comex_u12m,
    COALESCE(RCC2.max_atraso_competencia_descuentos_u12m,-1) max_atraso_competencia_descuentos_u12m,
    COALESCE(RCC2.max_atraso_competencia_directas_otros_u12m,-1) max_atraso_competencia_directas_otros_u12m,
    COALESCE(RCC2.max_atraso_competencia_factoring_u12m,-1) max_atraso_competencia_factoring_u12m,
    COALESCE(RCC2.max_atraso_competencia_prestamos_ajustados_u12m,-1) max_atraso_competencia_prestamos_ajustados_u12m,
    COALESCE(RCC2.max_atraso_competencia_prestamos_general_u12m,-1) max_atraso_competencia_prestamos_general_u12m,
    COALESCE(RCC2.max_atraso_competencia_leasing_u12m,-1) max_atraso_competencia_leasing_u12m,
    COALESCE(RCC2.max_atraso_competencia_tar_credito_u12m,-1) max_atraso_competencia_tar_credito_u12m,
    COALESCE(RCC2.max_atraso_competencia_inmuebeles_u12m,-1) max_atraso_competencia_inmuebeles_u12m,
    COALESCE(RCC2.max_atraso_competencia_fin_bienes_u12m,-1) max_atraso_competencia_fin_bienes_u12m,
    COALESCE(RCC2.max_atraso_competencia_impulso_u12m,-1) max_atraso_competencia_impulso_u12m,
    COALESCE(RCC2.max_atraso_competencia_reactiva_u12m,-1) max_atraso_competencia_reactiva_u12m,
    COALESCE(RCC2.max_atraso_competencia_reprog_fae_u12m,-1) max_atraso_competencia_reprog_fae_u12m,
    COALESCE(RCC2.max_atraso_competencia_reprog_emergencia_u12m,-1) max_atraso_competencia_reprog_emergencia_u12m,
    COALESCE(RCC2.max_atraso_competencia_coloc_directas_ajustado_u12m,-1) max_atraso_competencia_coloc_directas_ajustado_u12m,
    COALESCE(RCC2.max_atraso_competencia_coloc_directas_general_u12m,-1) max_atraso_competencia_coloc_directas_general_u12m,
    --COALESCE(RCC2.max_atraso_competencia_inmobilario_u12m,-1) max_atraso_competencia_inmobilario_u12m,
    COALESCE(RCC2.max_atraso_competencia_tipo_producto_impulso_u12m,-1) max_atraso_competencia_tipo_producto_impulso_u12m,
    COALESCE(RCC2.max_atraso_competencia_tipo_producto_reactiva_u12m,-1) max_atraso_competencia_tipo_producto_reactiva_u12m,
    COALESCE(RCC2.max_atraso_competencia_tipo_producto_reprog_u12m,-1) max_atraso_competencia_tipo_producto_reprog_u12m,
    COALESCE(RCC2.max_atraso_competencia_tipo_producto_fae_u12m,-1) max_atraso_competencia_tipo_producto_fae_u12m,
    COALESCE(RCC2.max_atraso_competencia_tipo_producto_pae_u12,-1) max_atraso_competencia_tipo_producto_pae_u12,
    COALESCE(RCC2.avg_nro_entidades_cajas_u6m,0) avg_nro_entidades_cajas_u6m,
    COALESCE(RCC2.avg_nro_entidades_financieras_u6m,0) avg_nro_entidades_financieras_u6m,
    COALESCE(RCC2.avg_nro_entidades_bancos_u6m,0) avg_nro_entidades_bancos_u6m,
    COALESCE(RCC2.avg_nro_tipos_entidades_u6m,0) avg_nro_tipos_entidades_u6m,
    COALESCE(RCC2.avg_nro_tipos_productos_u6m,0) avg_nro_tipos_productos_u6m,
    COALESCE(RCC2.avg_nro_productos_u6m,0) avg_nro_productos_u6m,
    COALESCE(RCC2.avg_nro_coloc_directas_normal_u6m,0) avg_nro_coloc_directas_normal_u6m,
    --COALESCE(RCC2.avg_nro_inmobilario_normal_u6m,0) avg_nro_inmobilario_normal_u6m,
    COALESCE(RCC2.avg_nro_reactiva_normal_u6m,0) avg_nro_reactiva_normal_u6m,
    COALESCE(RCC2.avg_nro_impulso_normal_u6m,0) avg_nro_impulso_normal_u6m,
    COALESCE(RCC2.avg_nro_reprogramado_normal_u6m,0) avg_nro_reprogramado_normal_u6m,
    COALESCE(RCC2.avg_nro_reg_coloc_directas_bancos_ajustado_u6m,0) avg_nro_reg_coloc_directas_bancos_ajustado_u6m,
    COALESCE(RCC2.avg_nro_reg_coloc_directas_bancos_general_u6m,0) avg_nro_reg_coloc_directas_bancos_general_u6m,
    COALESCE(RCC2.avg_nro_reg_coloc_indirectas_bancos_u6m,0) avg_nro_reg_coloc_indirectas_bancos_u6m,
    --COALESCE(RCC2.avg_nro_reg_inmobilario_bancos_u6m,0) avg_nro_reg_inmobilario_bancos_u6m,
    COALESCE(RCC2.avg_nro_reg_garantias_bancos_u6m,0) avg_nro_reg_garantias_bancos_u6m,
    COALESCE(RCC2.avg_nro_reg_reprogramados_bancos_u6m,0) avg_nro_reg_reprogramados_bancos_u6m,
    COALESCE(RCC2.avg_nro_reg_coloc_directas_cajas_financ_ajustado_u6m,0) avg_nro_reg_coloc_directas_cajas_financ_ajustado_u6m,
    COALESCE(RCC2.avg_nro_reg_coloc_directas_cajas_financ_general_u6m,0) avg_nro_reg_coloc_directas_cajas_financ_general_u6m,
    COALESCE(RCC2.avg_nro_reg_coloc_indirectas_cajas_financ_u6m,0) avg_nro_reg_coloc_indirectas_cajas_financ_u6m,
    COALESCE(RCC2.avg_nro_reg_garantias_cajas_financ_u6m,0) avg_nro_reg_garantias_cajas_financ_u6m,
    COALESCE(RCC2.avg_nro_reg_reprogramados_cajas_financ_u6m,0) avg_nro_reg_reprogramados_cajas_financ_u6m,
    COALESCE(RCC2.avg_saldo_castigado_amt_u6m,0) avg_saldo_castigado_amt_u6m,
    COALESCE(RCC2.avg_saldo_coloc_directas_amt_general_u6m,0) avg_saldo_coloc_directas_amt_general_u6m,
    COALESCE(RCC2.avg_saldo_coloc_directas_ajustado_amt_u6m,0) avg_saldo_coloc_directas_ajustado_amt_u6m,
    COALESCE(RCC2.avg_saldo_coloc_indirectas_amt_u6m,0) avg_saldo_coloc_indirectas_amt_u6m,
    COALESCE(RCC2.avg_saldo_garantias_amt_u6m,0) avg_saldo_garantias_amt_u6m,
    --COALESCE(RCC2.avg_saldo_inmobilario_amt_u6m,0) avg_saldo_inmobilario_amt_u6m,
    COALESCE(RCC2.avg_saldo_reprogramados_amt_u6m,0) avg_saldo_reprogramados_amt_u6m,
    COALESCE(RCC2.avg_saldo_coloc_directas_general_vig_amt_u6m,0) avg_saldo_coloc_directas_general_vig_amt_u6m,
    COALESCE(RCC2.avg_saldo_coloc_directas_vig_amt_u6m,0) avg_saldo_coloc_directas_vig_amt_u6m,
    --COALESCE(RCC2.avg_saldo_inmobilario_vig_amt_u6m,0) avg_saldo_inmobilario_vig_amt_u6m,
    COALESCE(RCC2.avg_saldo_reprogramados_vig_amt_u6m,0) avg_saldo_reprogramados_vig_amt_u6m,
    COALESCE(RCC2.avg_saldo_coloc_directas_general_no_vig_amt_u6m,0) avg_saldo_coloc_directas_general_no_vig_amt_u6m,
    COALESCE(RCC2.avg_saldo_coloc_directas_no_vig_amt_u6m,0) avg_saldo_coloc_directas_no_vig_amt_u6m,
    --COALESCE(RCC2.avg_saldo_inmobilario_no_vig_amt_u6m,0) avg_saldo_inmobilario_no_vig_amt_u6m,
    COALESCE(RCC2.avg_saldo_reprogramados_no_vig_amt_u6m,0) avg_saldo_reprogramados_no_vig_amt_u6m,
    COALESCE(RCC2.avg_saldo_garantias_otros_amt_u6m,0) avg_saldo_garantias_otros_amt_u6m,
    COALESCE(RCC2.avg_saldo_garantias_no_pref_amt_u6m,0) avg_saldo_garantias_no_pref_amt_u6m,
    COALESCE(RCC2.avg_saldo_garantias_hipo_amt_u6m,0) avg_saldo_garantias_hipo_amt_u6m,
    COALESCE(RCC2.avg_saldo_garantias_auto_amt_u6m,0) avg_saldo_garantias_auto_amt_u6m,
    COALESCE(RCC2.avg_saldo_garantias_warrants_amt_u6m,0) avg_saldo_garantias_warrants_amt_u6m,
    COALESCE(RCC2.avg_saldo_garantias_pref_amt_u6m,0) avg_saldo_garantias_pref_amt_u6m,
    COALESCE(RCC2.avg_saldo_bco_castigado_amt_u6m,0) avg_saldo_bco_castigado_amt_u6m,
    COALESCE(RCC2.avg_saldo_bco_coloc_directas_amt_general_u6m,0) avg_saldo_bco_coloc_directas_amt_general_u6m,
    COALESCE(RCC2.avg_saldo_bco_coloc_directas_ajustado_amt_u6m,0) avg_saldo_bco_coloc_directas_ajustado_amt_u6m,
    COALESCE(RCC2.avg_saldo_bco_coloc_indirectas_amt_u6m,0) avg_saldo_bco_coloc_indirectas_amt_u6m,
    COALESCE(RCC2.avg_saldo_bco_garantias_amt_u6m,0) avg_saldo_bco_garantias_amt_u6m,
    --COALESCE(RCC2.avg_saldo_bco_inmobilario_amt_u6m,0) avg_saldo_bco_inmobilario_amt_u6m,
    COALESCE(RCC2.avg_saldo_bco_reprogramados_amt_u6m,0) avg_saldo_bco_reprogramados_amt_u6m,
    COALESCE(RCC2.avg_saldo_bco_coloc_directas_general_vig_amt_u6m,0) avg_saldo_bco_coloc_directas_general_vig_amt_u6m,
    COALESCE(RCC2.avg_saldo_bco_coloc_directas_vig_amt_u6m,0) avg_saldo_bco_coloc_directas_vig_amt_u6m,
    --COALESCE(RCC2.avg_saldo_bco_inmobilario_vig_amt_u6m,0) avg_saldo_bco_inmobilario_vig_amt_u6m,
    COALESCE(RCC2.avg_saldo_bco_reprogramados_vig_amt_u6m,0) avg_saldo_bco_reprogramados_vig_amt_u6m,
    COALESCE(RCC2.avg_saldo_bco_coloc_directas_general_no_vig_amt_u6m,0) avg_saldo_bco_coloc_directas_general_no_vig_amt_u6m,
    COALESCE(RCC2.avg_saldo_bco_coloc_directas_no_vig_amt_u6m,0) avg_saldo_bco_coloc_directas_no_vig_amt_u6m,
    --COALESCE(RCC2.avg_saldo_bco_inmobilario_no_vig_amt_u6m,0) avg_saldo_bco_inmobilario_no_vig_amt_u6m,
    COALESCE(RCC2.avg_saldo_bco_reprogramados_no_vig_amt_u6m,0) avg_saldo_bco_reprogramados_no_vig_amt_u6m,
    COALESCE(RCC2.avg_saldo_bco_garantias_otros_amt_u6m,0) avg_saldo_bco_garantias_otros_amt_u6m,
    COALESCE(RCC2.avg_saldo_bco_garantias_no_pref_amt_u6m,0) avg_saldo_bco_garantias_no_pref_amt_u6m,
    COALESCE(RCC2.avg_saldo_bco_garantias_hipo_amt_u6m,0) avg_saldo_bco_garantias_hipo_amt_u6m,
    COALESCE(RCC2.avg_saldo_bco_garantias_auto_amt_u6m,0) avg_saldo_bco_garantias_auto_amt_u6m,
    COALESCE(RCC2.avg_saldo_bco_garantias_warrants_amt_u6m,0) avg_saldo_bco_garantias_warrants_amt_u6m,
    COALESCE(RCC2.avg_saldo_bco_garantias_pref_amt_u6m,0) avg_saldo_bco_garantias_pref_amt_u6m,
    COALESCE(RCC2.avg_saldo_ibk_castigado_amt_u6m,0) avg_saldo_ibk_castigado_amt_u6m,
    COALESCE(RCC2.avg_saldo_ibk_coloc_directas_amt_general_u6m,0) avg_saldo_ibk_coloc_directas_amt_general_u6m,
    COALESCE(RCC2.avg_saldo_ibk_coloc_directas_ajustado_amt_u6m,0) avg_saldo_ibk_coloc_directas_ajustado_amt_u6m,
    COALESCE(RCC2.avg_saldo_ibk_coloc_indirectas_amt_u6m,0) avg_saldo_ibk_coloc_indirectas_amt_u6m,
    COALESCE(RCC2.avg_saldo_ibk_garantias_amt_u6m,0) avg_saldo_ibk_garantias_amt_u6m,
    --COALESCE(RCC2.avg_saldo_ibk_inmobilario_amt_u6m,0) avg_saldo_ibk_inmobilario_amt_u6m,
    COALESCE(RCC2.avg_saldo_ibk_reprogramados_amt_u6m,0) avg_saldo_ibk_reprogramados_amt_u6m,
    COALESCE(RCC2.avg_saldo_ibk_coloc_directas_general_vig_amt_u6m,0) avg_saldo_ibk_coloc_directas_general_vig_amt_u6m,
    COALESCE(RCC2.avg_saldo_ibk_coloc_directas_vig_amt_u6m,0) avg_saldo_ibk_coloc_directas_vig_amt_u6m,
   -- COALESCE(RCC2.avg_saldo_ibk_inmobilario_vig_amt_u6m,0) avg_saldo_ibk_inmobilario_vig_amt_u6m,
    COALESCE(RCC2.avg_saldo_ibk_reprogramados_vig_amt_u6m,0) avg_saldo_ibk_reprogramados_vig_amt_u6m,
    COALESCE(RCC2.avg_saldo_ibk_coloc_directas_general_no_vig_amt_u6m,0) avg_saldo_ibk_coloc_directas_general_no_vig_amt_u6m,
    COALESCE(RCC2.avg_saldo_ibk_coloc_directas_no_vig_amt_u6m,0) avg_saldo_ibk_coloc_directas_no_vig_amt_u6m,
    --COALESCE(RCC2.avg_saldo_ibk_inmobilario_no_vig_amt_u6m,0) avg_saldo_ibk_inmobilario_no_vig_amt_u6m,
    COALESCE(RCC2.avg_saldo_ibk_reprogramados_no_vig_amt_u6m,0) avg_saldo_ibk_reprogramados_no_vig_amt_u6m,
    COALESCE(RCC2.avg_saldo_ibk_garantias_otros_amt_u6m,0) avg_saldo_ibk_garantias_otros_amt_u6m,
    COALESCE(RCC2.avg_saldo_ibk_garantias_no_pref_amt_u6m,0) avg_saldo_ibk_garantias_no_pref_amt_u6m,
    COALESCE(RCC2.avg_saldo_ibk_garantias_hipo_amt_u6m,0) avg_saldo_ibk_garantias_hipo_amt_u6m,
    COALESCE(RCC2.avg_saldo_ibk_garantias_auto_amt_u6m,0) avg_saldo_ibk_garantias_auto_amt_u6m,
    COALESCE(RCC2.avg_saldo_ibk_garantias_warrants_amt_u6m,0) avg_saldo_ibk_garantias_warrants_amt_u6m,
    COALESCE(RCC2.avg_saldo_ibk_garantias_pref_amt_u6m,0) avg_saldo_ibk_garantias_pref_amt_u6m,
    COALESCE(RCC2.avg_saldo_bco_no_ibk_castigado_amt_u6m,0) avg_saldo_bco_no_ibk_castigado_amt_u6m,
    COALESCE(RCC2.avg_saldo_bco_no_ibk_coloc_directas_amt_general_u6m,0) avg_saldo_bco_no_ibk_coloc_directas_amt_general_u6m,
    COALESCE(RCC2.avg_saldo_bco_no_ibk_coloc_directas_ajustado_amt_u6m,0) avg_saldo_bco_no_ibk_coloc_directas_ajustado_amt_u6m,
    COALESCE(RCC2.avg_saldo_bco_no_ibk_coloc_indirectas_amt_u6m,0) avg_saldo_bco_no_ibk_coloc_indirectas_amt_u6m,
    COALESCE(RCC2.avg_saldo_bco_no_ibk_garantias_amt_u6m,0) avg_saldo_bco_no_ibk_garantias_amt_u6m,
    COALESCE(RCC2.avg_saldo_bco_no_ibk_inmobilario_amt_u6m,0) avg_saldo_bco_no_ibk_inmobilario_amt_u6m,
    COALESCE(RCC2.avg_saldo_bco_no_ibk_reprogramados_amt_u6m,0) avg_saldo_bco_no_ibk_reprogramados_amt_u6m,
    COALESCE(RCC2.avg_saldo_bco_no_ibk_coloc_directas_general_vig_amt_u6m,0) avg_saldo_bco_no_ibk_coloc_directas_general_vig_amt_u6m,
    COALESCE(RCC2.avg_saldo_bco_no_ibk_coloc_directas_vig_amt_u6m,0) avg_saldo_bco_no_ibk_coloc_directas_vig_amt_u6m,
    COALESCE(RCC2.avg_saldo_bco_no_ibk_inmobilario_vig_amt_u6m,0) avg_saldo_bco_no_ibk_inmobilario_vig_amt_u6m,
    COALESCE(RCC2.avg_saldo_bco_no_ibk_reprogramados_vig_amt_u6m,0) avg_saldo_bco_no_ibk_reprogramados_vig_amt_u6m,
    COALESCE(RCC2.avg_saldo_bco_no_ibk_coloc_directas_general_no_vig_amt_u6m,0) avg_saldo_bco_no_ibk_coloc_directas_general_no_vig_amt_u6m,
    COALESCE(RCC2.avg_saldo_bco_no_ibk_coloc_directas_no_vig_amt_u6m,0) avg_saldo_bco_no_ibk_coloc_directas_no_vig_amt_u6m,
    COALESCE(RCC2.avg_saldo_bco_no_ibk_inmobilario_no_vig_amt_u6m,0) avg_saldo_bco_no_ibk_inmobilario_no_vig_amt_u6m,
    COALESCE(RCC2.avg_saldo_bco_no_ibk_reprogramados_no_vig_amt_u6m,0) avg_saldo_bco_no_ibk_reprogramados_no_vig_amt_u6m,
    COALESCE(RCC2.avg_saldo_bco_no_ibk_garantias_otros_amt_u6m,0) avg_saldo_bco_no_ibk_garantias_otros_amt_u6m,
    COALESCE(RCC2.avg_saldo_bco_no_ibk_garantias_no_pref_amt_u6m,0) avg_saldo_bco_no_ibk_garantias_no_pref_amt_u6m,
    COALESCE(RCC2.avg_saldo_bco_no_ibk_garantias_hipo_amt_u6m,0) avg_saldo_bco_no_ibk_garantias_hipo_amt_u6m,
    COALESCE(RCC2.avg_saldo_bco_no_ibk_garantias_auto_amt_u6m,0) avg_saldo_bco_no_ibk_garantias_auto_amt_u6m,
    COALESCE(RCC2.avg_saldo_bco_no_ibk_garantias_warrants_amt_u6m,0) avg_saldo_bco_no_ibk_garantias_warrants_amt_u6m,
    COALESCE(RCC2.avg_saldo_bco_no_ibk_garantias_pref_amt_u6m,0) avg_saldo_bco_no_ibk_garantias_pref_amt_u6m,
    COALESCE(RCC2.avg_saldo_cajas_financ_castigado_amt_u6m,0) avg_saldo_cajas_financ_castigado_amt_u6m,
    COALESCE(RCC2.avg_saldo_cajas_financ_coloc_directas_amt_general_u6m,0) avg_saldo_cajas_financ_coloc_directas_amt_general_u6m,
    COALESCE(RCC2.avg_saldo_cajas_financ_coloc_directas_ajustado_amt_u6m,0) avg_saldo_cajas_financ_coloc_directas_ajustado_amt_u6m,
    COALESCE(RCC2.avg_saldo_cajas_financ_coloc_indirectas_amt_u6m,0) avg_saldo_cajas_financ_coloc_indirectas_amt_u6m,
    COALESCE(RCC2.avg_saldo_cajas_financ_garantias_amt_u6m,0) avg_saldo_cajas_financ_garantias_amt_u6m,
    COALESCE(RCC2.avg_saldo_cajas_financ_reprogramados_amt_u6m,0) avg_saldo_cajas_financ_reprogramados_amt_u6m,
    COALESCE(RCC2.avg_saldo_cajas_financ_coloc_directas_general_vig_amt_u6m,0) avg_saldo_cajas_financ_coloc_directas_general_vig_amt_u6m,
    COALESCE(RCC2.avg_saldo_cajas_financ_coloc_directas_vig_amt_u6m,0) avg_saldo_cajas_financ_coloc_directas_vig_amt_u6m,
    COALESCE(RCC2.avg_saldo_cajas_financ_reprogramados_vig_amt_u6m,0) avg_saldo_cajas_financ_reprogramados_vig_amt_u6m,
    COALESCE(RCC2.avg_saldo_cajas_financ_coloc_directas_general_no_vig_amt_u6m,0) avg_saldo_cajas_financ_coloc_directas_general_no_vig_amt_u6m,
    COALESCE(RCC2.avg_saldo_cajas_financ_coloc_directas_no_vig_amt_u6m,0) avg_saldo_cajas_financ_coloc_directas_no_vig_amt_u6m,
    COALESCE(RCC2.avg_saldo_cajas_financ_reprogramados_no_vig_amt_u6m,0) avg_saldo_cajas_financ_reprogramados_no_vig_amt_u6m,
    COALESCE(RCC2.avg_saldo_cajas_financ_garantias_otros_amt_u6m,0) avg_saldo_cajas_financ_garantias_otros_amt_u6m,
    COALESCE(RCC2.avg_saldo_cajas_financ_garantias_no_pref_amt_u6m,0) avg_saldo_cajas_financ_garantias_no_pref_amt_u6m,
    COALESCE(RCC2.avg_saldo_cajas_financ_garantias_hipo_amt_u6m,0) avg_saldo_cajas_financ_garantias_hipo_amt_u6m,
    COALESCE(RCC2.avg_saldo_cajas_financ_garantias_auto_amt_u6m,0) avg_saldo_cajas_financ_garantias_auto_amt_u6m,
    COALESCE(RCC2.avg_saldo_cajas_financ_garantias_warrants_amt_u6m,0) avg_saldo_cajas_financ_garantias_warrants_amt_u6m,
    COALESCE(RCC2.avg_saldo_cajas_financ_garantias_pref_amt_u6m,0) avg_saldo_cajas_financ_garantias_pref_amt_u6m,
    COALESCE(RCC2.avg_saldo_competencia_castigado_amt_u6m,0) avg_saldo_competencia_castigado_amt_u6m,
    COALESCE(RCC2.avg_saldo_competencia_coloc_directas_amt_general_u6m,0) avg_saldo_competencia_coloc_directas_amt_general_u6m,
    COALESCE(RCC2.avg_saldo_competencia_coloc_directas_ajustado_amt_u6m,0) avg_saldo_competencia_coloc_directas_ajustado_amt_u6m,
    COALESCE(RCC2.avg_saldo_competencia_coloc_indirectas_amt_u6m,0) avg_saldo_competencia_coloc_indirectas_amt_u6m,
    COALESCE(RCC2.avg_saldo_competencia_garantias_amt_u6m,0) avg_saldo_competencia_garantias_amt_u6m,
    --COALESCE(RCC2.avg_saldo_competencia_inmobilario_amt_u6m,0) avg_saldo_competencia_inmobilario_amt_u6m,
    COALESCE(RCC2.avg_saldo_competencia_reprogramados_amt_u6m,0) avg_saldo_competencia_reprogramados_amt_u6m,
    COALESCE(RCC2.avg_saldo_competencia_coloc_directas_general_vig_amt_u6m,0) avg_saldo_competencia_coloc_directas_general_vig_amt_u6m,
    COALESCE(RCC2.avg_saldo_competencia_coloc_directas_vig_amt_u6m,0) avg_saldo_competencia_coloc_directas_vig_amt_u6m,
    --COALESCE(RCC2.avg_saldo_competencia_inmobilario_vig_amt_u6m,0) avg_saldo_competencia_inmobilario_vig_amt_u6m,
    COALESCE(RCC2.avg_saldo_competencia_reprogramados_vig_amt_u6m,0) avg_saldo_competencia_reprogramados_vig_amt_u6m,
    COALESCE(RCC2.avg_saldo_competencia_coloc_directas_general_no_vig_amt_u6m,0) avg_saldo_competencia_coloc_directas_general_no_vig_amt_u6m,
    COALESCE(RCC2.avg_saldo_competencia_coloc_directas_no_vig_amt_u6m,0) avg_saldo_competencia_coloc_directas_no_vig_amt_u6m,
    --COALESCE(RCC2.avg_saldo_competencia_inmobilario_no_vig_amt_u6m,0) avg_saldo_competencia_inmobilario_no_vig_amt_u6m,
    COALESCE(RCC2.avg_saldo_competencia_reprogramados_no_vig_amt_u6m,0) avg_saldo_competencia_reprogramados_no_vig_amt_u6m,
    COALESCE(RCC2.avg_saldo_competencia_garantias_otros_amt_u6m,0) avg_saldo_competencia_garantias_otros_amt_u6m,
    COALESCE(RCC2.avg_saldo_competencia_garantias_no_pref_amt_u6m,0) avg_saldo_competencia_garantias_no_pref_amt_u6m,
    COALESCE(RCC2.avg_saldo_competencia_garantias_hipo_amt_u6m,0) avg_saldo_competencia_garantias_hipo_amt_u6m,
    COALESCE(RCC2.avg_saldo_competencia_garantias_auto_amt_u6m,0) avg_saldo_competencia_garantias_auto_amt_u6m,
    COALESCE(RCC2.avg_saldo_competencia_garantias_warrants_amt_u6m,0) avg_saldo_competencia_garantias_warrants_amt_u6m,
    COALESCE(RCC2.avg_saldo_competencia_garantias_pref_amt_u6m,0) avg_saldo_competencia_garantias_pref_amt_u6m,
    COALESCE(RCC2.avg_saldo_sobregiros_amt_u6m,0) avg_saldo_sobregiros_amt_u6m,
    COALESCE(RCC2.avg_saldo_comex_amt_u6m,0) avg_saldo_comex_amt_u6m,
    COALESCE(RCC2.avg_saldo_descuentos_amt_u6m,0) avg_saldo_descuentos_amt_u6m,
    COALESCE(RCC2.avg_saldo_directas_otros_amt_u6m,0) avg_saldo_directas_otros_amt_u6m,
    COALESCE(RCC2.avg_saldo_factoring_amt_u6m,0) avg_saldo_factoring_amt_u6m,
    COALESCE(RCC2.avg_saldo_prestamos_ajustados_amt_u6m,0) avg_saldo_prestamos_ajustados_amt_u6m,
    COALESCE(RCC2.avg_saldo_prestamos_general_amt_u6m,0) avg_saldo_prestamos_general_amt_u6m,
    COALESCE(RCC2.avg_saldo_leasing_amt_u6m,0) avg_saldo_leasing_amt_u6m,
    COALESCE(RCC2.avg_saldo_tar_credito_amt_u6m,0) avg_saldo_tar_credito_amt_u6m,
    COALESCE(RCC2.avg_saldo_carta_credito_amt_u6m,0) avg_saldo_carta_credito_amt_u6m,
    COALESCE(RCC2.avg_saldo_carta_fianza_amt_u6m,0) avg_saldo_carta_fianza_amt_u6m,
    COALESCE(RCC2.avg_saldo_deriv_instr_amt_u6m,0) avg_saldo_deriv_instr_amt_u6m,
    COALESCE(RCC2.avg_saldo_deriv_moenda_amt_u6m,0) avg_saldo_deriv_moenda_amt_u6m,
    COALESCE(RCC2.avg_saldo_fin_fae_amt_u6m,0) avg_saldo_fin_fae_amt_u6m,
    COALESCE(RCC2.avg_saldo_gar_fae_amt_u6m,0) avg_saldo_gar_fae_amt_u6m,
    COALESCE(RCC2.avg_saldo_gar_impulso_amt_u6m,0) avg_saldo_gar_impulso_amt_u6m,
    COALESCE(RCC2.avg_saldo_gar_pae_amt_u6m,0) avg_saldo_gar_pae_amt_u6m,
    COALESCE(RCC2.avg_saldo_gar_reactiva_amt_u6m,0) avg_saldo_gar_reactiva_amt_u6m,
    COALESCE(RCC2.avg_saldo_inmuebeles_amt_u6m,0) avg_saldo_inmuebeles_amt_u6m,
    COALESCE(RCC2.avg_saldo_fin_bienes_amt_u6m,0) avg_saldo_fin_bienes_amt_u6m,
    COALESCE(RCC2.avg_saldo_impulso_amt_u6m,0) avg_saldo_impulso_amt_u6m,
    COALESCE(RCC2.avg_saldo_reactiva_amt_u6m,0) avg_saldo_reactiva_amt_u6m,
    COALESCE(RCC2.avg_saldo_reprog_fae_amt_u6m,0) avg_saldo_reprog_fae_amt_u6m,
    COALESCE(RCC2.avg_saldo_reprog_emergencia_amt_u6m,0) avg_saldo_reprog_emergencia_amt_u6m,
    COALESCE(RCC2.avg_saldo_vig_sobregiros_amt_u6m,0) avg_saldo_vig_sobregiros_amt_u6m,
    COALESCE(RCC2.avg_saldo_vig_comex_amt_u6m,0) avg_saldo_vig_comex_amt_u6m,
    COALESCE(RCC2.avg_saldo_vig_descuentos_amt_u6m,0) avg_saldo_vig_descuentos_amt_u6m,
    COALESCE(RCC2.avg_saldo_vig_directas_otros_amt_u6m,0) avg_saldo_vig_directas_otros_amt_u6m,
    COALESCE(RCC2.avg_saldo_vig_factoring_amt_u6m,0) avg_saldo_vig_factoring_amt_u6m,
    COALESCE(RCC2.avg_saldo_vig_prestamos_general_amt_u6m,0) avg_saldo_vig_prestamos_general_amt_u6m,
    COALESCE(RCC2.avg_saldo_vig_leasing_amt_u6m,0) avg_saldo_vig_leasing_amt_u6m,
    COALESCE(RCC2.avg_saldo_vig_tar_credito_amt_u6m,0) avg_saldo_vig_tar_credito_amt_u6m,
    COALESCE(RCC2.avg_saldo_vig_impulso_amt_u6m,0) avg_saldo_vig_impulso_amt_u6m,
    COALESCE(RCC2.avg_saldo_vig_reactiva_amt_u6m,0) avg_saldo_vig_reactiva_amt_u6m,
    COALESCE(RCC2.avg_saldo_vig_reprog_fae_amt_u6m,0) avg_saldo_vig_reprog_fae_amt_u6m,
    COALESCE(RCC2.avg_saldo_vig_reprog_emergencia_amt_u6m,0) avg_saldo_vig_reprog_emergencia_amt_u6m,
    COALESCE(RCC2.avg_saldo_no_vig_sobregiros_amt_u6m,0) avg_saldo_no_vig_sobregiros_amt_u6m,
    COALESCE(RCC2.avg_saldo_no_vig_comex_amt_u6m,0) avg_saldo_no_vig_comex_amt_u6m,
    COALESCE(RCC2.avg_saldo_no_vig_directas_otros_amt_u6m,0) avg_saldo_no_vig_directas_otros_amt_u6m,
    COALESCE(RCC2.avg_saldo_no_vig_factoring_amt_u6m,0) avg_saldo_no_vig_factoring_amt_u6m,
    COALESCE(RCC2.avg_saldo_no_vig_prestamos_general_amt_u6m,0) avg_saldo_no_vig_prestamos_general_amt_u6m,
    COALESCE(RCC2.avg_saldo_no_vig_leasing_amt_u6m,0) avg_saldo_no_vig_leasing_amt_u6m,
    COALESCE(RCC2.avg_saldo_no_vig_tar_credito_amt_u6m,0) avg_saldo_no_vig_tar_credito_amt_u6m,
    COALESCE(RCC2.avg_saldo_no_vig_fin_bienes_amt_u6m,0) avg_saldo_no_vig_fin_bienes_amt_u6m,
    COALESCE(RCC2.avg_saldo_no_vig_impulso_amt_u6m,0) avg_saldo_no_vig_impulso_amt_u6m,
    COALESCE(RCC2.avg_saldo_no_vig_reactiva_amt_u6m,0) avg_saldo_no_vig_reactiva_amt_u6m,
    COALESCE(RCC2.avg_saldo_no_vig_reprog_fae_amt_u6m,0) avg_saldo_no_vig_reprog_fae_amt_u6m,
    COALESCE(RCC2.avg_saldo_no_vig_reprog_emergencia_amt_u6m,0) avg_saldo_no_vig_reprog_emergencia_amt_u6m,
    COALESCE(RCC2.max_atraso_sobregiros_u6m,-1) max_atraso_sobregiros_u6m,
    COALESCE(RCC2.max_atraso_comex_u6m,-1) max_atraso_comex_u6m,
    COALESCE(RCC2.max_atraso_descuentos_u6m,-1) max_atraso_descuentos_u6m,
    COALESCE(RCC2.max_atraso_directas_otros_u6m,-1) max_atraso_directas_otros_u6m,
    COALESCE(RCC2.max_atraso_factoring_u6m,-1) max_atraso_factoring_u6m,
    COALESCE(RCC2.max_atraso_prestamos_ajustados_u6m,-1) max_atraso_prestamos_ajustados_u6m,
    COALESCE(RCC2.max_atraso_prestamos_general_u6m,-1) max_atraso_prestamos_general_u6m,
    COALESCE(RCC2.max_atraso_leasing_u6m,-1) max_atraso_leasing_u6m,
    COALESCE(RCC2.max_atraso_tar_credito_u6m,-1) max_atraso_tar_credito_u6m,
    COALESCE(RCC2.max_atraso_inmuebeles_u6m,-1) max_atraso_inmuebeles_u6m,
    COALESCE(RCC2.max_atraso_fin_bienes_u6m,-1) max_atraso_fin_bienes_u6m,
    COALESCE(RCC2.max_atraso_impulso_u6m,-1) max_atraso_impulso_u6m,
    COALESCE(RCC2.max_atraso_reactiva_u6m,-1) max_atraso_reactiva_u6m,
    COALESCE(RCC2.max_atraso_reprog_fae_u6m,-1) max_atraso_reprog_fae_u6m,
    COALESCE(RCC2.max_atraso_reprog_emergencia_u6m,-1) max_atraso_reprog_emergencia_u6m,
    COALESCE(RCC2.max_atraso_coloc_directas_ajustado_u6m,-1) max_atraso_coloc_directas_ajustado_u6m,
    COALESCE(RCC2.max_atraso_coloc_directas_general_u6m,-1) max_atraso_coloc_directas_general_u6m,
    --COALESCE(RCC2.max_atraso_inmobilario_u6m,-1) max_atraso_inmobilario_u6m,
    COALESCE(RCC2.max_atraso_tipo_producto_impulso_u6m,-1) max_atraso_tipo_producto_impulso_u6m,
    COALESCE(RCC2.max_atraso_tipo_producto_reactiva_u6m,-1) max_atraso_tipo_producto_reactiva_u6m,
    COALESCE(RCC2.max_atraso_tipo_producto_reprog_u6m,-1) max_atraso_tipo_producto_reprog_u6m,
    COALESCE(RCC2.max_atraso_tipo_producto_fae_u6m,-1) max_atraso_tipo_producto_fae_u6m,
    COALESCE(RCC2.max_atraso_tipo_producto_pae_u6m,-1) max_atraso_tipo_producto_pae_u6m,
    COALESCE(RCC2.max_atraso_bco_sobregiros_u6m,-1) max_atraso_bco_sobregiros_u6m,
    COALESCE(RCC2.max_atraso_bco_comex_u6m,-1) max_atraso_bco_comex_u6m,
    COALESCE(RCC2.max_atraso_bco_descuentos_u6m,-1) max_atraso_bco_descuentos_u6m,
    COALESCE(RCC2.max_atraso_bco_directas_otros_u6m,-1) max_atraso_bco_directas_otros_u6m,
    COALESCE(RCC2.max_atraso_bco_factoring_u6m,-1) max_atraso_bco_factoring_u6m,
    COALESCE(RCC2.max_atraso_bco_prestamos_ajustados_u6m,-1) max_atraso_bco_prestamos_ajustados_u6m,
    COALESCE(RCC2.max_atraso_bco_prestamos_general_u6m,-1) max_atraso_bco_prestamos_general_u6m,
    COALESCE(RCC2.max_atraso_bco_leasing_u6m,-1) max_atraso_bco_leasing_u6m,
    COALESCE(RCC2.max_atraso_bco_tar_credito_u6m,-1) max_atraso_bco_tar_credito_u6m,
    COALESCE(RCC2.max_atraso_bco_inmuebeles_u6m,-1) max_atraso_bco_inmuebeles_u6m,
    COALESCE(RCC2.max_atraso_bco_fin_bienes_u6m,-1) max_atraso_bco_fin_bienes_u6m,
    COALESCE(RCC2.max_atraso_bco_impulso_u6m,-1) max_atraso_bco_impulso_u6m,
    COALESCE(RCC2.max_atraso_bco_reactiva_u6m,-1) max_atraso_bco_reactiva_u6m,
    COALESCE(RCC2.max_atraso_bco_reprog_fae_u6m,-1) max_atraso_bco_reprog_fae_u6m,
    COALESCE(RCC2.max_atraso_bco_reprog_emergencia_u6m,-1) max_atraso_bco_reprog_emergencia_u6m,
    COALESCE(RCC2.max_atraso_bco_coloc_directas_ajustado_u6m,-1) max_atraso_bco_coloc_directas_ajustado_u6m,
    COALESCE(RCC2.max_atraso_bco_coloc_directas_general_u6m,-1) max_atraso_bco_coloc_directas_general_u6m,
    --COALESCE(RCC2.max_atraso_bco_inmobilario_u6m,-1) max_atraso_bco_inmobilario_u6m,
    COALESCE(RCC2.max_atraso_bco_tipo_producto_impulso_u6m,-1) max_atraso_bco_tipo_producto_impulso_u6m,
    COALESCE(RCC2.max_atraso_bco_tipo_producto_reactiva_u6m,-1) max_atraso_bco_tipo_producto_reactiva_u6m,
    COALESCE(RCC2.max_atraso_bco_tipo_producto_reprog_u6m,-1) max_atraso_bco_tipo_producto_reprog_u6m,
    COALESCE(RCC2.max_atraso_bco_tipo_producto_fae_u6m,-1) max_atraso_bco_tipo_producto_fae_u6m,
    COALESCE(RCC2.max_atraso_bco_tipo_producto_pae_u6m,-1) max_atraso_bco_tipo_producto_pae_u6m,
    COALESCE(RCC2.max_atraso_ibk_sobregiros_u6m,-1) max_atraso_ibk_sobregiros_u6m,
    COALESCE(RCC2.max_atraso_ibk_comex_u6m,-1) max_atraso_ibk_comex_u6m,
    COALESCE(RCC2.max_atraso_ibk_descuentos_u6m,-1) max_atraso_ibk_descuentos_u6m,
    COALESCE(RCC2.max_atraso_ibk_directas_otros_u6m,-1) max_atraso_ibk_directas_otros_u6m,
    COALESCE(RCC2.max_atraso_ibk_factoring_u6m,-1) max_atraso_ibk_factoring_u6m,
    COALESCE(RCC2.max_atraso_ibk_prestamos_ajustados_u6m,-1) max_atraso_ibk_prestamos_ajustados_u6m,
    COALESCE(RCC2.max_atraso_ibk_prestamos_general_u6m,-1) max_atraso_ibk_prestamos_general_u6m,
    COALESCE(RCC2.max_atraso_ibk_leasing_u6m,-1) max_atraso_ibk_leasing_u6m,
    COALESCE(RCC2.max_atraso_ibk_tar_credito_u6m,-1) max_atraso_ibk_tar_credito_u6m,
    COALESCE(RCC2.max_atraso_ibk_inmuebeles_u6m,-1) max_atraso_ibk_inmuebeles_u6m,
    COALESCE(RCC2.max_atraso_ibk_fin_bienes_u6m,-1) max_atraso_ibk_fin_bienes_u6m,
    COALESCE(RCC2.max_atraso_ibk_impulso_u6m,-1) max_atraso_ibk_impulso_u6m,
    COALESCE(RCC2.max_atraso_ibk_reactiva_u6m,-1) max_atraso_ibk_reactiva_u6m,
    COALESCE(RCC2.max_atraso_ibk_reprog_fae_u6m,-1) max_atraso_ibk_reprog_fae_u6m,
    COALESCE(RCC2.max_atraso_ibk_reprog_emergencia_u6m,-1) max_atraso_ibk_reprog_emergencia_u6m,
    COALESCE(RCC2.max_atraso_ibk_coloc_directas_ajustado_u6m,-1) max_atraso_ibk_coloc_directas_ajustado_u6m,
    COALESCE(RCC2.max_atraso_ibk_coloc_directas_general_u6m,-1) max_atraso_ibk_coloc_directas_general_u6m,
    --COALESCE(RCC2.max_atraso_ibk_inmobilario_u6m,-1) max_atraso_ibk_inmobilario_u6m,
    COALESCE(RCC2.max_atraso_ibk_tipo_producto_impulso_u6m,-1) max_atraso_ibk_tipo_producto_impulso_u6m,
    COALESCE(RCC2.max_atraso_ibk_tipo_producto_reactiva_u6m,-1) max_atraso_ibk_tipo_producto_reactiva_u6m,
    COALESCE(RCC2.max_atraso_ibk_tipo_producto_reprog_u6m,-1) max_atraso_ibk_tipo_producto_reprog_u6m,
    COALESCE(RCC2.max_atraso_ibk_tipo_producto_fae_u6m,-1) max_atraso_ibk_tipo_producto_fae_u6m,
    COALESCE(RCC2.max_atraso_ibk_tipo_producto_pae_u6m,-1) max_atraso_ibk_tipo_producto_pae_u6m,
    COALESCE(RCC2.max_atraso_competencia_sobregiros_u6m,-1) max_atraso_competencia_sobregiros_u6m,
    COALESCE(RCC2.max_atraso_competencia_comex_u6m,-1) max_atraso_competencia_comex_u6m,
    COALESCE(RCC2.max_atraso_competencia_descuentos_u6m,-1) max_atraso_competencia_descuentos_u6m,
    COALESCE(RCC2.max_atraso_competencia_directas_otros_u6m,-1) max_atraso_competencia_directas_otros_u6m,
    COALESCE(RCC2.max_atraso_competencia_factoring_u6m,-1) max_atraso_competencia_factoring_u6m,
    COALESCE(RCC2.max_atraso_competencia_prestamos_ajustados_u6m,-1) max_atraso_competencia_prestamos_ajustados_u6m,
    COALESCE(RCC2.max_atraso_competencia_prestamos_general_u6m,-1) max_atraso_competencia_prestamos_general_u6m,
    COALESCE(RCC2.max_atraso_competencia_leasing_u6m,-1) max_atraso_competencia_leasing_u6m,
    COALESCE(RCC2.max_atraso_competencia_tar_credito_u6m,-1) max_atraso_competencia_tar_credito_u6m,
    COALESCE(RCC2.max_atraso_competencia_inmuebeles_u6m,-1) max_atraso_competencia_inmuebeles_u6m,
    COALESCE(RCC2.max_atraso_competencia_fin_bienes_u6m,-1) max_atraso_competencia_fin_bienes_u6m,
    COALESCE(RCC2.max_atraso_competencia_impulso_u6m,-1) max_atraso_competencia_impulso_u6m,
    COALESCE(RCC2.max_atraso_competencia_reactiva_u6m,-1) max_atraso_competencia_reactiva_u6m,
    COALESCE(RCC2.max_atraso_competencia_reprog_fae_u6m,-1) max_atraso_competencia_reprog_fae_u6m,
    COALESCE(RCC2.max_atraso_competencia_reprog_emergencia_u6m,-1) max_atraso_competencia_reprog_emergencia_u6m,
    COALESCE(RCC2.max_atraso_competencia_coloc_directas_ajustado_u6m,-1) max_atraso_competencia_coloc_directas_ajustado_u6m,
    COALESCE(RCC2.max_atraso_competencia_coloc_directas_general_u6m,-1) max_atraso_competencia_coloc_directas_general_u6m,
    --COALESCE(RCC2.max_atraso_competencia_inmobilario_u6m,-1) max_atraso_competencia_inmobilario_u6m,
    COALESCE(RCC2.max_atraso_competencia_tipo_producto_impulso_u6m,-1) max_atraso_competencia_tipo_producto_impulso_u6m,
    COALESCE(RCC2.max_atraso_competencia_tipo_producto_reactiva_u6m,-1) max_atraso_competencia_tipo_producto_reactiva_u6m,
    COALESCE(RCC2.max_atraso_competencia_tipo_producto_reprog_u6m,-1) max_atraso_competencia_tipo_producto_reprog_u6m,
    COALESCE(RCC2.max_atraso_competencia_tipo_producto_fae_u6m,-1) max_atraso_competencia_tipo_producto_fae_u6m,
    COALESCE(RCC2.max_atraso_competencia_tipo_producto_pae_u6m,-1) max_atraso_competencia_tipo_producto_pae_u6m,


    COALESCE(RCC3.tendencia_nro_entidades,0) tendencia_nro_entidades,
    COALESCE(RCC3.tendencia_nro_tipos_productos,0) tendencia_nro_tipos_productos,
    COALESCE(RCC3.tendencia_nro_productos,0) tendencia_nro_productos,
    COALESCE(RCC3.tendencia_nro_coloc_directas_normal,0) tendencia_nro_coloc_directas_normal,
    COALESCE(RCC3.tendencia_nro_reg_coloc_directas_bancos_general,0) tendencia_nro_reg_coloc_directas_bancos_general,
    COALESCE(RCC3.tendencia_nro_reg_coloc_indirectas_bancos,0) tendencia_nro_reg_coloc_indirectas_bancos,
    COALESCE(RCC3.tendencia_saldo_castigado_amt,0) tendencia_saldo_castigado_amt,
    COALESCE(RCC3.tendencia_saldo_coloc_directas_amt_general,0) tendencia_saldo_coloc_directas_amt_general,
    COALESCE(RCC3.tendencia_saldo_coloc_indirectas_amt,0) tendencia_saldo_coloc_indirectas_amt,
    COALESCE(RCC3.tendencia_saldo_garantias_amt,0) tendencia_saldo_garantias_amt,
    COALESCE(RCC3.tendencia_saldo_reprogramados_amt,0) tendencia_saldo_reprogramados_amt,
    --COALESCE(RCC3.tendencia_saldo_inmobilario_amt,0) tendencia_saldo_inmobilario_amt,
    COALESCE(RCC3.tendencia_saldo_coloc_directas_general_no_vig_amt,0) tendencia_saldo_coloc_directas_general_no_vig_amt,
    COALESCE(RCC3.tendencia_saldo_coloc_directas_no_vig_amt,0) tendencia_saldo_coloc_directas_no_vig_amt,
    COALESCE(RCC3.tendencia_saldo_garantias_pref_amt,0) tendencia_saldo_garantias_pref_amt,
    COALESCE(RCC3.tendencia_saldo_ibk_castigado_amt,0) tendencia_saldo_ibk_castigado_amt,
    COALESCE(RCC3.tendencia_saldo_ibk_coloc_directas_amt_general,0) tendencia_saldo_ibk_coloc_directas_amt_general,
    COALESCE(RCC3.tendencia_saldo_ibk_coloc_indirectas_amt,0) tendencia_saldo_ibk_coloc_indirectas_amt,
    COALESCE(RCC3.tendencia_saldo_ibk_garantias_amt,0) tendencia_saldo_ibk_garantias_amt,
    COALESCE(RCC3.tendencia_saldo_ibk_reprogramados_amt,0) tendencia_saldo_ibk_reprogramados_amt,
    --COALESCE(RCC3.tendencia_saldo_ibk_inmobilario_amt,0) tendencia_saldo_ibk_inmobilario_amt,
    COALESCE(RCC3.tendencia_saldo_bco_no_ibk_castigado_amt,0) tendencia_saldo_bco_no_ibk_castigado_amt,
    COALESCE(RCC3.tendencia_saldo_bco_no_ibk_coloc_directas_amt_general,0) tendencia_saldo_bco_no_ibk_coloc_directas_amt_general,
    COALESCE(RCC3.tendencia_saldo_bco_no_ibk_coloc_indirectas_amt,0) tendencia_saldo_bco_no_ibk_coloc_indirectas_amt,
    COALESCE(RCC3.tendencia_saldo_bco_no_ibk_garantias_amt,0) tendencia_saldo_bco_no_ibk_garantias_amt,
    COALESCE(RCC3.tendencia_saldo_bco_no_ibk_reprogramados_amt,0) tendencia_saldo_bco_no_ibk_reprogramados_amt,
    --COALESCE(RCC3.tendencia_saldo_bco_no_ibk_inmobilario_amt,0) tendencia_saldo_bco_no_ibk_inmobilario_amt,
    COALESCE(RCC3.tendencia_saldo_cajas_financ_castigado_amt,0) tendencia_saldo_cajas_financ_castigado_amt,
    COALESCE(RCC3.tendencia_saldo_cajas_financ_coloc_directas_amt_general,0) tendencia_saldo_cajas_financ_coloc_directas_amt_general,
    COALESCE(RCC3.tendencia_saldo_cajas_financ_coloc_indirectas_amt,0) tendencia_saldo_cajas_financ_coloc_indirectas_amt,
    COALESCE(RCC3.tendencia_saldo_cajas_financ_garantias_amt,0) tendencia_saldo_cajas_financ_garantias_amt,
    COALESCE(RCC3.tendencia_saldo_cajas_financ_reprogramados_amt,0) tendencia_saldo_cajas_financ_reprogramados_amt,
    COALESCE(RCC3.tendencia_saldo_sobregiros_amt,0) tendencia_saldo_sobregiros_amt,
    COALESCE(RCC3.tendencia_saldo_comex_amt,0) tendencia_saldo_comex_amt,
    COALESCE(RCC3.tendencia_saldo_descuentos_amt,0) tendencia_saldo_descuentos_amt,
    COALESCE(RCC3.tendencia_saldo_directas_otros_amt,0) tendencia_saldo_directas_otros_amt,
    COALESCE(RCC3.tendencia_saldo_factoring_amt,0) tendencia_saldo_factoring_amt,
    COALESCE(RCC3.tendencia_saldo_prestamos_ajustados_amt,0) tendencia_saldo_prestamos_ajustados_amt,
    COALESCE(RCC3.tendencia_saldo_prestamos_general_amt,0) tendencia_saldo_prestamos_general_amt,
    COALESCE(RCC3.tendencia_saldo_leasing_amt,0) tendencia_saldo_leasing_amt,
    COALESCE(RCC3.tendencia_saldo_tar_credito_amt,0) tendencia_saldo_tar_credito_amt,
    COALESCE(RCC3.tendencia_saldo_carta_credito_amt,0) tendencia_saldo_carta_credito_amt,
    COALESCE(RCC3.tendencia_saldo_carta_fianza_amt,0) tendencia_saldo_carta_fianza_amt,
    COALESCE(RCC3.tendencia_saldo_inmuebeles_amt,0) tendencia_saldo_inmuebeles_amt,
    COALESCE(RCC3.tendencia_saldo_fin_bienes_amt,0) tendencia_saldo_fin_bienes_amt,
    COALESCE(RCC3.tendencia_saldo_impulso_amt,0) tendencia_saldo_impulso_amt,
    COALESCE(RCC3.tendencia_saldo_reactiva_amt,0) tendencia_saldo_reactiva_amt,
    COALESCE(RCC3.tendencia_saldo_reprog_fae_amt,0) tendencia_saldo_reprog_fae_amt,
    COALESCE(RCC3.tendencia_saldo_reprog_emergencia_amt,0) tendencia_saldo_reprog_emergencia_amt,
    COALESCE(RCC3.flg_recien_saldo_sobregiros_amt_u3m,0) flg_recien_saldo_sobregiros_amt_u3m,
    COALESCE(RCC3.flg_ya_no_saldo_sobregiros_amt_u3m,0) flg_ya_no_saldo_sobregiros_amt_u3m,
    COALESCE(RCC3.flg_recien_saldo_comex_amt_u3m,0) flg_recien_saldo_comex_amt_u3m,
    COALESCE(RCC3.flg_ya_no_saldo_comex_amt_u3m,0) flg_ya_no_saldo_comex_amt_u3m,
    COALESCE(RCC3.flg_recien_saldo_descuentos_amt_u3m,0) flg_recien_saldo_descuentos_amt_u3m,
    COALESCE(RCC3.flg_ya_no_saldo_descuentos_amt_u3m,0) flg_ya_no_saldo_descuentos_amt_u3m,
    COALESCE(RCC3.flg_recien_saldo_directas_otros_amt_u3m,0) flg_recien_saldo_directas_otros_amt_u3m,
    COALESCE(RCC3.flg_ya_no_saldo_directas_otros_amt_u3m,0) flg_ya_no_saldo_directas_otros_amt_u3m,
    COALESCE(RCC3.flg_recien_saldo_factoring_amt_u3m,0) flg_recien_saldo_factoring_amt_u3m,
    COALESCE(RCC3.flg_ya_no_saldo_factoring_amt_u3m,0) flg_ya_no_saldo_factoring_amt_u3m,
    COALESCE(RCC3.flg_recien_saldo_prestamos_ajustados_amt_u3m,0) flg_recien_saldo_prestamos_ajustados_amt_u3m,
    COALESCE(RCC3.flg_ya_no_saldo_prestamos_ajustados_amt_u3m,0) flg_ya_no_saldo_prestamos_ajustados_amt_u3m,
    COALESCE(RCC3.flg_recien_saldo_prestamos_general_amt_u3m,0) flg_recien_saldo_prestamos_general_amt_u3m,
    COALESCE(RCC3.flg_ya_no_saldo_prestamos_general_amt_u3m,0) flg_ya_no_saldo_prestamos_general_amt_u3m,
    COALESCE(RCC3.flg_recien_saldo_leasing_amt_u3m,0) flg_recien_saldo_leasing_amt_u3m,
    COALESCE(RCC3.flg_ya_no_saldo_leasing_amt_u3m,0) flg_ya_no_saldo_leasing_amt_u3m,
    COALESCE(RCC3.flg_recien_saldo_tar_credito_amt_u3m,0) flg_recien_saldo_tar_credito_amt_u3m,
    COALESCE(RCC3.flg_ya_no_saldo_tar_credito_amt_u3m,0) flg_ya_no_saldo_tar_credito_amt_u3m,
    COALESCE(RCC3.flg_recien_saldo_carta_credito_amt_u3m,0) flg_recien_saldo_carta_credito_amt_u3m,
    COALESCE(RCC3.flg_ya_no_saldo_carta_credito_amt_u3m,0) flg_ya_no_saldo_carta_credito_amt_u3m,
    COALESCE(RCC3.flg_recien_saldo_carta_fianza_amt_u3m,0) flg_recien_saldo_carta_fianza_amt_u3m,
    COALESCE(RCC3.flg_ya_no_saldo_carta_fianza_amt_u3m,0) flg_ya_no_saldo_carta_fianza_amt_u3m,
    COALESCE(RCC3.flg_recien_saldo_fin_fae_amt_u3m,0) flg_recien_saldo_fin_fae_amt_u3m,
    COALESCE(RCC3.flg_ya_no_saldo_fin_fae_amt_u3m,0) flg_ya_no_saldo_fin_fae_amt_u3m,
    COALESCE(RCC3.flg_recien_saldo_gar_fae_amt_u3m,0) flg_recien_saldo_gar_fae_amt_u3m,
    COALESCE(RCC3.flg_ya_no_saldo_gar_fae_amt_u3m,0) flg_ya_no_saldo_gar_fae_amt_u3m,
    COALESCE(RCC3.flg_recien_saldo_gar_impulso_amt_u3m,0) flg_recien_saldo_gar_impulso_amt_u3m,
    COALESCE(RCC3.flg_ya_no_saldo_gar_impulso_amt_u3m,0) flg_ya_no_saldo_gar_impulso_amt_u3m,
    COALESCE(RCC3.flg_recien_saldo_gar_pae_amt_u3m,0) flg_recien_saldo_gar_pae_amt_u3m,
    COALESCE(RCC3.flg_ya_no_saldo_gar_pae_amt_u3m,0) flg_ya_no_saldo_gar_pae_amt_u3m,
    COALESCE(RCC3.flg_recien_saldo_gar_reactiva_amt_u3m,0) flg_recien_saldo_gar_reactiva_amt_u3m,
    COALESCE(RCC3.flg_ya_no_saldo_gar_reactiva_amt_u3m,0) flg_ya_no_saldo_gar_reactiva_amt_u3m,
    COALESCE(RCC3.flg_recien_saldo_inmuebeles_amt_u3m,0) flg_recien_saldo_inmuebeles_amt_u3m,
    COALESCE(RCC3.flg_ya_no_saldo_inmuebeles_amt_u3m,0) flg_ya_no_saldo_inmuebeles_amt_u3m,
    COALESCE(RCC3.flg_recien_saldo_fin_bienes_amt_u3m,0) flg_recien_saldo_fin_bienes_amt_u3m,
    COALESCE(RCC3.flg_ya_no_saldo_fin_bienes_amt_u3m,0) flg_ya_no_saldo_fin_bienes_amt_u3m,
    COALESCE(RCC3.flg_recien_saldo_impulso_amt_u3m,0) flg_recien_saldo_impulso_amt_u3m,
    COALESCE(RCC3.flg_ya_no_saldo_impulso_amt_u3m,0) flg_ya_no_saldo_impulso_amt_u3m,
    COALESCE(RCC3.flg_recien_saldo_reactiva_amt_u3m,0) flg_recien_saldo_reactiva_amt_u3m,
    COALESCE(RCC3.flg_ya_no_saldo_reactiva_amt_u3m,0) flg_ya_no_saldo_reactiva_amt_u3m,
    COALESCE(RCC3.flg_recien_saldo_reprog_fae_amt_u3m,0) flg_recien_saldo_reprog_fae_amt_u3m,
    COALESCE(RCC3.flg_ya_no_saldo_reprog_fae_amt_u3m,0) flg_ya_no_saldo_reprog_fae_amt_u3m,
    COALESCE(RCC3.flg_recien_saldo_reprog_emergencia_amt_u3m,0) flg_recien_saldo_reprog_emergencia_amt_u3m,
    COALESCE(RCC3.flg_ya_no_saldo_reprog_emergencia_amt_u3m,0) flg_ya_no_saldo_reprog_emergencia_amt_u3m,
    COALESCE(RCC3.flg_tiene_saldo_sobregiros_amt_u6m,0) flg_tiene_saldo_sobregiros_amt_u6m,
    COALESCE(RCC3.flg_tiene_saldo_comex_amt_u6m,0) flg_tiene_saldo_comex_amt_u6m,
    COALESCE(RCC3.flg_tiene_saldo_descuentos_amt_u6m,0) flg_tiene_saldo_descuentos_amt_u6m,
    COALESCE(RCC3.flg_tiene_saldo_directas_otros_amt_u6m,0) flg_tiene_saldo_directas_otros_amt_u6m,
    COALESCE(RCC3.flg_tiene_saldo_factoring_amt_u6m,0) flg_tiene_saldo_factoring_amt_u6m,
    COALESCE(RCC3.flg_tiene_saldo_prestamos_ajustados_amt_u6m,0) flg_tiene_saldo_prestamos_ajustados_amt_u6m,
    COALESCE(RCC3.flg_tiene_saldo_prestamos_general_amt_u6m,0) flg_tiene_saldo_prestamos_general_amt_u6m,
    COALESCE(RCC3.flg_tiene_saldo_leasing_amt_u6m,0) flg_tiene_saldo_leasing_amt_u6m,
    COALESCE(RCC3.flg_tiene_saldo_tar_credito_amt_u6m,0) flg_tiene_saldo_tar_credito_amt_u6m,
    COALESCE(RCC3.flg_tiene_saldo_carta_credito_amt_u6m,0) flg_tiene_saldo_carta_credito_amt_u6m,
    COALESCE(RCC3.flg_tiene_saldo_carta_fianza_amt_u6m,0) flg_tiene_saldo_carta_fianza_amt_u6m,
    COALESCE(RCC3.flg_tiene_saldo_fin_fae_amt_u6m,0) flg_tiene_saldo_fin_fae_amt_u6m,
    COALESCE(RCC3.flg_tiene_saldo_gar_fae_amt_u6m,0) flg_tiene_saldo_gar_fae_amt_u6m,
    COALESCE(RCC3.flg_tiene_saldo_gar_impulso_amt_u6m,0) flg_tiene_saldo_gar_impulso_amt_u6m,
    COALESCE(RCC3.flg_tiene_saldo_gar_pae_amt_u6m,0) flg_tiene_saldo_gar_pae_amt_u6m,
    COALESCE(RCC3.flg_tiene_saldo_gar_reactiva_amt_u6m,0) flg_tiene_saldo_gar_reactiva_amt_u6m,
    COALESCE(RCC3.flg_tiene_saldo_inmuebeles_amt_u6m,0) flg_tiene_saldo_inmuebeles_amt_u6m,
    COALESCE(RCC3.flg_tiene_saldo_fin_bienes_amt_u6m,0) flg_tiene_saldo_fin_bienes_amt_u6m,
    COALESCE(RCC3.flg_tiene_saldo_impulso_amt_u6m,0) flg_tiene_saldo_impulso_amt_u6m,
    COALESCE(RCC3.flg_tiene_saldo_reactiva_amt_u6m,0) flg_tiene_saldo_reactiva_amt_u6m,
    COALESCE(RCC3.flg_tiene_saldo_reprog_fae_amt_u6m,0) flg_tiene_saldo_reprog_fae_amt_u6m,
    COALESCE(RCC3.flg_tiene_saldo_reprog_emergencia_amt_u6m,0) flg_tiene_saldo_reprog_emergencia_amt_u6m,
    COALESCE(RCC3.tendencia_max_atraso_sobregiros,0) tendencia_max_atraso_sobregiros,
    COALESCE(RCC3.tendencia_max_atraso_comex,0) tendencia_max_atraso_comex,
    COALESCE(RCC3.tendencia_max_atraso_descuentos,0) tendencia_max_atraso_descuentos,
    COALESCE(RCC3.tendencia_max_atraso_directas_otros,0) tendencia_max_atraso_directas_otros,
    COALESCE(RCC3.tendencia_max_atraso_factoring,0) tendencia_max_atraso_factoring,
    COALESCE(RCC3.tendencia_max_atraso_prestamos_ajustados,0) tendencia_max_atraso_prestamos_ajustados,
    COALESCE(RCC3.tendencia_max_atraso_prestamos_general,0) tendencia_max_atraso_prestamos_general,
    COALESCE(RCC3.tendencia_max_atraso_leasing,0) tendencia_max_atraso_leasing,
    COALESCE(RCC3.tendencia_max_atraso_tar_credito,0) tendencia_max_atraso_tar_credito,
    COALESCE(RCC3.tendencia_max_atraso_inmuebeles,0) tendencia_max_atraso_inmuebeles,
    COALESCE(RCC3.tendencia_max_atraso_fin_bienes,0) tendencia_max_atraso_fin_bienes,
    COALESCE(RCC3.tendencia_max_atraso_reactiva,0) tendencia_max_atraso_reactiva,
    COALESCE(RCC3.tendencia_max_atraso_reprog_fae,0) tendencia_max_atraso_reprog_fae,
    COALESCE(RCC3.tendencia_max_atraso_coloc_directas_ajustado,0) tendencia_max_atraso_coloc_directas_ajustado,
    COALESCE(RCC3.tendencia_max_atraso_coloc_directas_general,0) tendencia_max_atraso_coloc_directas_general,
    --COALESCE(RCC3.tendencia_max_atraso_inmobilario,0) tendencia_max_atraso_inmobilario,
    COALESCE(RCC3.tendencia_max_atraso_tipo_producto_impulso,0) tendencia_max_atraso_tipo_producto_impulso,
    COALESCE(RCC3.tendencia_max_atraso_tipo_producto_reactiva,0) tendencia_max_atraso_tipo_producto_reactiva,
    COALESCE(RCC3.tendencia_max_atraso_tipo_producto_reprog,0) tendencia_max_atraso_tipo_producto_reprog,
    COALESCE(RCC3.flg_recien_max_atraso_sobregiros_u3m,0) flg_recien_max_atraso_sobregiros_u3m,
    COALESCE(RCC3.flg_ya_no_max_atraso_sobregiros_u3m,0) flg_ya_no_max_atraso_sobregiros_u3m,
    COALESCE(RCC3.flg_recien_max_atraso_comex_u3m,0) flg_recien_max_atraso_comex_u3m,
    COALESCE(RCC3.flg_ya_no_max_atraso_comex_u3m,0) flg_ya_no_max_atraso_comex_u3m,
    COALESCE(RCC3.flg_recien_max_atraso_descuentos_u3m,0) flg_recien_max_atraso_descuentos_u3m,
    COALESCE(RCC3.flg_ya_no_max_atraso_descuentos_u3m,0) flg_ya_no_max_atraso_descuentos_u3m,
    COALESCE(RCC3.flg_recien_max_atraso_directas_otros_u3m,0) flg_recien_max_atraso_directas_otros_u3m,
    COALESCE(RCC3.flg_ya_no_max_atraso_directas_otros_u3m,0) flg_ya_no_max_atraso_directas_otros_u3m,
    COALESCE(RCC3.flg_recien_max_atraso_factoring_u3m,0) flg_recien_max_atraso_factoring_u3m,
    COALESCE(RCC3.flg_ya_no_max_atraso_factoring_u3m,0) flg_ya_no_max_atraso_factoring_u3m,
    COALESCE(RCC3.flg_recien_max_atraso_prestamos_ajustados_u3m,0) flg_recien_max_atraso_prestamos_ajustados_u3m,
    COALESCE(RCC3.flg_ya_no_max_atraso_prestamos_ajustados_u3m,0) flg_ya_no_max_atraso_prestamos_ajustados_u3m,
    COALESCE(RCC3.flg_recien_max_atraso_prestamos_general_u3m,0) flg_recien_max_atraso_prestamos_general_u3m,
    COALESCE(RCC3.flg_ya_no_max_atraso_prestamos_general_u3m,0) flg_ya_no_max_atraso_prestamos_general_u3m,
    COALESCE(RCC3.flg_recien_max_atraso_leasing_u3m,0) flg_recien_max_atraso_leasing_u3m,
    COALESCE(RCC3.flg_ya_no_max_atraso_leasing_u3m,0) flg_ya_no_max_atraso_leasing_u3m,
    COALESCE(RCC3.flg_recien_max_atraso_tar_credito_u3m,0) flg_recien_max_atraso_tar_credito_u3m,
    COALESCE(RCC3.flg_ya_no_max_atraso_tar_credito_u3m,0) flg_ya_no_max_atraso_tar_credito_u3m,
    COALESCE(RCC3.flg_recien_max_atraso_inmuebeles_u3m,0) flg_recien_max_atraso_inmuebeles_u3m,
    COALESCE(RCC3.flg_ya_no_max_atraso_inmuebeles_u3m,0) flg_ya_no_max_atraso_inmuebeles_u3m,
    COALESCE(RCC3.flg_recien_max_atraso_fin_bienes_u3m,0) flg_recien_max_atraso_fin_bienes_u3m,
    COALESCE(RCC3.flg_ya_no_max_atraso_fin_bienes_u3m,0) flg_ya_no_max_atraso_fin_bienes_u3m,
    COALESCE(RCC3.flg_recien_max_atraso_impulso_u3m,0) flg_recien_max_atraso_impulso_u3m,
    COALESCE(RCC3.flg_ya_no_max_atraso_impulso_u3m,0) flg_ya_no_max_atraso_impulso_u3m,
    COALESCE(RCC3.flg_recien_max_atraso_reactiva_u3m,0) flg_recien_max_atraso_reactiva_u3m,
    COALESCE(RCC3.flg_ya_no_max_atraso_reactiva_u3m,0) flg_ya_no_max_atraso_reactiva_u3m,
    COALESCE(RCC3.flg_recien_max_atraso_reprog_fae_u3m,0) flg_recien_max_atraso_reprog_fae_u3m,
    COALESCE(RCC3.flg_ya_no_max_atraso_reprog_fae_u3m,0) flg_ya_no_max_atraso_reprog_fae_u3m,
    COALESCE(RCC3.flg_recien_max_atraso_reprog_emergencia_u3m,0) flg_recien_max_atraso_reprog_emergencia_u3m,
    COALESCE(RCC3.flg_ya_no_max_atraso_reprog_emergencia_u3m,0) flg_ya_no_max_atraso_reprog_emergencia_u3m,
    COALESCE(RCC3.flg_tiene_max_atraso_sobregiros_u6m,0) flg_tiene_max_atraso_sobregiros_u6m,
    COALESCE(RCC3.flg_tiene_max_atraso_comex_u6m,0) flg_tiene_max_atraso_comex_u6m,
    COALESCE(RCC3.flg_tiene_max_atraso_descuentos_u6m,0) flg_tiene_max_atraso_descuentos_u6m,
    COALESCE(RCC3.flg_tiene_max_atraso_directas_otros_u6m,0) flg_tiene_max_atraso_directas_otros_u6m,
    COALESCE(RCC3.flg_tiene_max_atraso_factoring_u6m,0) flg_tiene_max_atraso_factoring_u6m,
    COALESCE(RCC3.flg_tiene_max_atraso_prestamos_ajustados_u6m,0) flg_tiene_max_atraso_prestamos_ajustados_u6m,
    COALESCE(RCC3.flg_tiene_max_atraso_prestamos_general_u6m,0) flg_tiene_max_atraso_prestamos_general_u6m,
    COALESCE(RCC3.flg_tiene_max_atraso_leasing_u6m,0) flg_tiene_max_atraso_leasing_u6m,
    COALESCE(RCC3.flg_tiene_max_atraso_tar_credito_u6m,0) flg_tiene_max_atraso_tar_credito_u6m,
    COALESCE(RCC3.flg_tiene_max_atraso_inmuebeles_u6m,0) flg_tiene_max_atraso_inmuebeles_u6m,
    COALESCE(RCC3.flg_tiene_max_atraso_fin_bienes_u6m,0) flg_tiene_max_atraso_fin_bienes_u6m,
    COALESCE(RCC3.flg_tiene_max_atraso_impulso_u6m,0) flg_tiene_max_atraso_impulso_u6m,
    COALESCE(RCC3.flg_tiene_max_atraso_reactiva_u6m,0) flg_tiene_max_atraso_reactiva_u6m,
    COALESCE(RCC3.flg_tiene_max_atraso_reprog_fae_u6m,0) flg_tiene_max_atraso_reprog_fae_u6m,
    


    COALESCE(RCC5.saldo_ajustado_amt_p12,0) saldo_ajustado_amt_p12,
    COALESCE(RCC5.flg_tiene_mas_de_12_meses_con_saldo_ajustado,0) flg_tiene_mas_de_12_meses_con_saldo_ajustado,
    COALESCE(RCC5.flg_tiene_mas_de_6_meses_con_saldo_ajustado,0) flg_tiene_mas_de_6_meses_con_saldo_ajustado,
    COALESCE(RCC5.flg_tiene_mas_de_3_meses_con_saldo_ajustado,0) flg_tiene_mas_de_3_meses_con_saldo_ajustado,
    COALESCE(RCC5.flg_tiene_1_meses_o_mas_con_saldo_ajustado,0) flg_tiene_1_meses_o_mas_con_saldo_ajustado,
    COALESCE(RCC5.ultima_variacion_saldo_ajustado,-9999999999) ultima_variacion_saldo_ajustado,
    COALESCE(RCC5.variacion_saldo_ajustado_p1m_p3m,-9999999999) variacion_saldo_ajustado_p1m_p3m,
    COALESCE(RCC5.variacion_saldo_ajustado_p3m_p6m,-9999999999) variacion_saldo_ajustado_p3m_p6m,
    COALESCE(RCC5.variacion_saldo_ajustado_presente_p3,-9999999999) variacion_saldo_ajustado_presente_p3,
    COALESCE(RCC5.variacion_saldo_ajustado_presente_p6,-9999999999) variacion_saldo_ajustado_presente_p6,
    COALESCE(RCC5.variacion_saldo_ajustado_presente_p12,-9999999999) variacion_saldo_ajustado_presente_p12,
    COALESCE(RCC5.div_saldo_ajustado_presente_p1,0) div_saldo_ajustado_presente_p1,
    COALESCE(RCC5.div_saldo_ajustado_presente_p3,0) div_saldo_ajustado_presente_p3,
    COALESCE(RCC5.div_saldo_ajustado_presente_p6,0) div_saldo_ajustado_presente_p6,
    COALESCE(RCC5.div_saldo_ajustado_presente_p12,0) div_saldo_ajustado_presente_p12,
    COALESCE(RCC5.div_saldo_ajustado_p1_presente,0) div_saldo_ajustado_p1_presente,
    COALESCE(RCC5.div_saldo_ajustado_p3_presente,0) div_saldo_ajustado_p3_presente,
    COALESCE(RCC5.div_saldo_ajustado_p6_presente,0) div_saldo_ajustado_p6_presente,
    COALESCE(RCC5.div_saldo_ajustado_p12_presente,0) div_saldo_ajustado_p12_presente,
    COALESCE(RCC5.flg_recien_menos_de_6_meses_con_saldo_ajustado,0) flg_recien_menos_de_6_meses_con_saldo_ajustado,
    COALESCE(RCC5.flg_no_tiene_saldo_ajustado_u12m,0) flg_no_tiene_saldo_ajustado_u12m,
    COALESCE(RCC5.flg_no_tiene_saldo_ajustado_u6m,0) flg_no_tiene_saldo_ajustado_u6m,
    COALESCE(RCC5.flg_no_tiene_saldo_ajustado_u3m,0) flg_no_tiene_saldo_ajustado_u3m,
    COALESCE(RCC5.flg_termino_prestamo_u6m,0) flg_termino_prestamo_u6m,
    COALESCE(RCC5.flg_termino_prestamo_u3m,0) flg_termino_prestamo_u3m,
    COALESCE(RCC5.flg_recien_3_meses_con_saldo_ajustado_u6m,0) flg_recien_3_meses_con_saldo_ajustado_u6m,
    COALESCE(RCC5.flg_recien_1_mes_con_saldo_ajustado_u6m,0) flg_recien_1_mes_con_saldo_ajustado_u6m,
    COALESCE(RCC5.monto_variacion_positiva_ult_rcc,0) monto_variacion_positiva_ult_rcc,
    COALESCE(RCC5.monto_variacion_positiva_p1m_p3m,0) monto_variacion_positiva_p1m_p3m,
    COALESCE(RCC5.monto_variacion_positiva_p3m_p6m,0) monto_variacion_positiva_p3m_p6m,
    COALESCE(RCC5.monto_variacion_positiva_p6m_p12m,0) monto_variacion_positiva_p6m_p12m,
    COALESCE(RCC5.monto_variacion_negativa_ult_rcc,0) monto_variacion_negativa_ult_rcc,
    COALESCE(RCC5.monto_variacion_negativa_p1m_p3m,0) monto_variacion_negativa_p1m_p3m,
    COALESCE(RCC5.monto_variacion_negativa_p3m_p6m,0) monto_variacion_negativa_p3m_p6m,
    COALESCE(RCC5.monto_variacion_negativa_p6m_p12m,0) monto_variacion_negativa_p6m_p12m,
    COALESCE(RCC5.flg_sin_variacion_ult_rcc,0) flg_sin_variacion_ult_rcc,
    COALESCE(RCC5.flg_sin_variacion_p1m_p3m,0) flg_sin_variacion_p1m_p3m,
    COALESCE(RCC5.flg_sin_variacion_p3m_p6m,0) flg_sin_variacion_p3m_p6m,
    COALESCE(RCC5.flg_sin_variacion_p6m_p12m,0) flg_sin_variacion_p6m_p12m,
    COALESCE(RCC5.flg_tiene_variacion_up10k_low30k_positiva_ult_rcc,0) flg_tiene_variacion_up10k_low30k_positiva_ult_rcc,
    COALESCE(RCC5.flg_tiene_variacion_up10k_low30k_positiva_p1m_p3m,0) flg_tiene_variacion_up10k_low30k_positiva_p1m_p3m,
    COALESCE(RCC5.flg_tiene_variacion_up10k_low30k_positiva_p3m_p6m,0) flg_tiene_variacion_up10k_low30k_positiva_p3m_p6m,
    COALESCE(RCC5.flg_tiene_variacion_up10k_low30k_positiva_p6m_p12m,0) flg_tiene_variacion_up10k_low30k_positiva_p6m_p12m,
    COALESCE(RCC5.flg_tiene_variacion_up30k_low180k_positiva_ult_rcc,0) flg_tiene_variacion_up30k_low180k_positiva_ult_rcc,
    COALESCE(RCC5.flg_tiene_variacion_up30k_low180k_positiva_p1m_p3m,0) flg_tiene_variacion_up30k_low180k_positiva_p1m_p3m,
    COALESCE(RCC5.flg_tiene_variacion_up30k_low180k_positiva_p3m_p6m,0) flg_tiene_variacion_up30k_low180k_positiva_p3m_p6m,
    COALESCE(RCC5.flg_tiene_variacion_up30k_low180k_positiva_p6m_p12m,0) flg_tiene_variacion_up30k_low180k_positiva_p6m_p12m,
    COALESCE(RCC5.flg_tiene_variacion_up180k_positiva_ult_rcc,0) flg_tiene_variacion_up180k_positiva_ult_rcc,
    COALESCE(RCC5.flg_tiene_variacion_up180k_positiva_p1m_p3m,0) flg_tiene_variacion_up180k_positiva_p1m_p3m,
    COALESCE(RCC5.flg_tiene_variacion_up180k_positiva_p3m_p6m,0) flg_tiene_variacion_up180k_positiva_p3m_p6m,
    COALESCE(RCC5.flg_tiene_variacion_up180k_positiva_p6m_p12m,0) flg_tiene_variacion_up180k_positiva_p6m_p12m,
    COALESCE(RCC5.flg_tiene_variacion_up180k_negativa_ult_rcc,0) flg_tiene_variacion_up180k_negativa_ult_rcc,
    COALESCE(RCC5.flg_tiene_variacion_up180k_negativa_p1m_p3m,0) flg_tiene_variacion_up180k_negativa_p1m_p3m,
    COALESCE(RCC5.flg_tiene_variacion_up180k_negativa_p3m_p6m,0) flg_tiene_variacion_up180k_negativa_p3m_p6m,
    COALESCE(RCC5.flg_tiene_variacion_up180k_negativa_p6m_p12m,0) flg_tiene_variacion_up180k_negativa_p6m_p12m,
    COALESCE(RCC5.sistema_finan_amt,0) sistema_finan_amt,
    COALESCE(RCC5.bancos_amt,0) bancos_amt,
    COALESCE(RCC5.bbva_amt,0) bbva_amt,
    COALESCE(RCC5.bcp_amt,0) bcp_amt,
    COALESCE(RCC5.bif_amt,0) bif_amt,
    COALESCE(RCC5.ibk_amt,0) ibk_amt,
    COALESCE(RCC5.scotia_amt,0) scotia_amt,
    COALESCE(RCC5.cajas_finan_amt,0) cajas_finan_amt,
    COALESCE(RCC5.otros_amt,0) otros_amt,
    COALESCE(RCC5.sistema_finan_directa_amt,0) sistema_finan_directa_amt,
    COALESCE(RCC5.bancos_directa_amt,0) bancos_directa_amt,
    COALESCE(RCC5.bbva_directa_amt,0) bbva_directa_amt,
    COALESCE(RCC5.bcp_directa_amt,0) bcp_directa_amt,
    COALESCE(RCC5.bif_directa_amt,0) bif_directa_amt,
    COALESCE(RCC5.ibk_directa_amt,0) ibk_directa_amt,
    COALESCE(RCC5.scotia_directa_amt,0) scotia_directa_amt,
    COALESCE(RCC5.cajas_finan_directa_amt,0) cajas_finan_directa_amt,
    COALESCE(RCC5.otros_directa_amt,0) otros_directa_amt,
    COALESCE(RCC5.sistema_finan_indirecta_amt,0) sistema_finan_indirecta_amt,
    COALESCE(RCC5.bancos_indirecta_amt,0) bancos_indirecta_amt,
    COALESCE(RCC5.bbva_indirecta_amt,0) bbva_indirecta_amt,
    COALESCE(RCC5.bcp_indirecta_amt,0) bcp_indirecta_amt,
    COALESCE(RCC5.bif_indirecta_amt,0) bif_indirecta_amt,
    COALESCE(RCC5.ibk_indirecta_amt,0) ibk_indirecta_amt,
    COALESCE(RCC5.scotia_indirecta_amt,0) scotia_indirecta_amt,
    COALESCE(RCC5.cajas_finan_indirecta_amt,0) cajas_finan_indirecta_amt,
    COALESCE(RCC5.otros_indirecta_amt,0) otros_indirecta_amt,
    COALESCE(RCC5.sistema_finan_m1_amt,0) sistema_finan_m1_amt,
    COALESCE(RCC5.bancos_m1_amt,0) bancos_m1_amt,
    COALESCE(RCC5.bbva_m1_amt,0) bbva_m1_amt,
    COALESCE(RCC5.bcp_m1_amt,0) bcp_m1_amt,
    COALESCE(RCC5.bif_m1_amt,0) bif_m1_amt,
    COALESCE(RCC5.ibk_m1_amt,0) ibk_m1_amt,
    COALESCE(RCC5.scotia_m1_amt,0) scotia_m1_amt,
    COALESCE(RCC5.cajas_finan_m1_amt,0) cajas_finan_m1_amt,
    COALESCE(RCC5.otros_m1_amt,0) otros_m1_amt,
    COALESCE(RCC5.sistema_finan_directa_m1_amt,0) sistema_finan_directa_m1_amt,
    COALESCE(RCC5.bancos_directa_m1_amt,0) bancos_directa_m1_amt,
    COALESCE(RCC5.bbva_directa_m1_amt,0) bbva_directa_m1_amt,
    COALESCE(RCC5.bcp_directa_m1_amt,0) bcp_directa_m1_amt,
    COALESCE(RCC5.bif_directa_m1_amt,0) bif_directa_m1_amt,
    COALESCE(RCC5.ibk_directa_m1_amt,0) ibk_directa_m1_amt,
    COALESCE(RCC5.scotia_directa_m1_amt,0) scotia_directa_m1_amt,
    COALESCE(RCC5.cajas_finan_directa_m1_amt,0) cajas_finan_directa_m1_amt,
    COALESCE(RCC5.otros_directa_m1_amt,0) otros_directa_m1_amt,
    COALESCE(RCC5.sistema_finan_indirecta_m1_amt,0) sistema_finan_indirecta_m1_amt,
    COALESCE(RCC5.bancos_indirecta_m1_amt,0) bancos_indirecta_m1_amt,
    COALESCE(RCC5.bbva_indirecta_m1_amt,0) bbva_indirecta_m1_amt,
    COALESCE(RCC5.bcp_indirecta_m1_amt,0) bcp_indirecta_m1_amt,
    COALESCE(RCC5.bif_indirecta_m1_amt,0) bif_indirecta_m1_amt,
    COALESCE(RCC5.ibk_indirecta_m1_amt,0) ibk_indirecta_m1_amt,
    COALESCE(RCC5.scotia_indirecta_m1_amt,0) scotia_indirecta_m1_amt,
    COALESCE(RCC5.cajas_finan_indirecta_m1_amt,0) cajas_finan_indirecta_m1_amt,
    COALESCE(RCC5.otros_indirecta_m1_amt,0) otros_indirecta_m1_amt,
    COALESCE(RCC5.sistema_finan_m12_amt,0) sistema_finan_m12_amt,
    COALESCE(RCC5.bancos_m12_amt,0) bancos_m12_amt,
    COALESCE(RCC5.bbva_m12_amt,0) bbva_m12_amt,
    COALESCE(RCC5.bcp_m12_amt,0) bcp_m12_amt,
    COALESCE(RCC5.bif_m12_amt,0) bif_m12_amt,
    COALESCE(RCC5.ibk_m12_amt,0) ibk_m12_amt,
    COALESCE(RCC5.scotia_m12_amt,0) scotia_m12_amt,
    COALESCE(RCC5.cajas_finan_m12_amt,0) cajas_finan_m12_amt,
    COALESCE(RCC5.otros_m12_amt,0) otros_m12_amt,
    COALESCE(RCC5.sistema_finan_directa_m12_amt,0) sistema_finan_directa_m12_amt,
    COALESCE(RCC5.bancos_directa_m12_amt,0) bancos_directa_m12_amt,
    COALESCE(RCC5.bbva_directa_m12_amt,0) bbva_directa_m12_amt,
    COALESCE(RCC5.bcp_directa_m12_amt,0) bcp_directa_m12_amt,
    COALESCE(RCC5.bif_directa_m12_amt,0) bif_directa_m12_amt,
    COALESCE(RCC5.ibk_directa_m12_amt,0) ibk_directa_m12_amt,
    COALESCE(RCC5.scotia_directa_m12_amt,0) scotia_directa_m12_amt,
    COALESCE(RCC5.cajas_finan_directa_m12_amt,0) cajas_finan_directa_m12_amt,
    COALESCE(RCC5.otros_directa_m12_amt,0) otros_directa_m12_amt,
    COALESCE(RCC5.sistema_finan_indirecta_m12_amt,0) sistema_finan_indirecta_m12_amt,
    COALESCE(RCC5.bancos_indirecta_m12_amt,0) bancos_indirecta_m12_amt,
    COALESCE(RCC5.bbva_indirecta_m12_amt,0) bbva_indirecta_m12_amt,
    COALESCE(RCC5.bcp_indirecta_m12_amt,0) bcp_indirecta_m12_amt,
    COALESCE(RCC5.bif_indirecta_m12_amt,0) bif_indirecta_m12_amt,
    COALESCE(RCC5.ibk_indirecta_m12_amt,0) ibk_indirecta_m12_amt,
    COALESCE(RCC5.scotia_indirecta_m12_amt,0) scotia_indirecta_m12_amt,
    COALESCE(RCC5.cajas_finan_indirecta_m12_amt,0) cajas_finan_indirecta_m12_amt,
    COALESCE(RCC5.otros_indirecta_m12_amt,0) otros_indirecta_m12_amt,
    COALESCE(RCC5.sistema_finan_ytd_amt,0) sistema_finan_ytd_amt,
    COALESCE(RCC5.bancos_ytd_amt,0) bancos_ytd_amt,
    COALESCE(RCC5.bbva_ytd_amt,0) bbva_ytd_amt,
    COALESCE(RCC5.bcp_ytd_amt,0) bcp_ytd_amt,
    COALESCE(RCC5.bif_ytd_amt,0) bif_ytd_amt,
    COALESCE(RCC5.ibk_ytd_amt,0) ibk_ytd_amt,
    COALESCE(RCC5.scotia_ytd_amt,0) scotia_ytd_amt,
    COALESCE(RCC5.cajas_finan_ytd_amt,0) cajas_finan_ytd_amt,
    COALESCE(RCC5.otros_ytd_amt,0) otros_ytd_amt,
    COALESCE(RCC5.sistema_finan_directa_ytd_amt,0) sistema_finan_directa_ytd_amt,
    COALESCE(RCC5.bancos_directa_ytd_amt,0) bancos_directa_ytd_amt,
    COALESCE(RCC5.bbva_directa_ytd_amt,0) bbva_directa_ytd_amt,
    COALESCE(RCC5.bcp_directa_ytd_amt,0) bcp_directa_ytd_amt,
    COALESCE(RCC5.bif_directa_ytd_amt,0) bif_directa_ytd_amt,
    COALESCE(RCC5.ibk_directa_ytd_amt,0) ibk_directa_ytd_amt,
    COALESCE(RCC5.scotia_directa_ytd_amt,0) scotia_directa_ytd_amt,
    COALESCE(RCC5.cajas_finan_directa_ytd_amt,0) cajas_finan_directa_ytd_amt,
    COALESCE(RCC5.otros_directa_ytd_amt,0) otros_directa_ytd_amt,
    COALESCE(RCC5.sistema_finan_indirecta_ytd_amt,0) sistema_finan_indirecta_ytd_amt,
    COALESCE(RCC5.bancos_indirecta_ytd_amt,0) bancos_indirecta_ytd_amt,
    COALESCE(RCC5.bbva_indirecta_ytd_amt,0) bbva_indirecta_ytd_amt,
    COALESCE(RCC5.bcp_indirecta_ytd_amt,0) bcp_indirecta_ytd_amt,
    COALESCE(RCC5.bif_indirecta_ytd_amt,0) bif_indirecta_ytd_amt,
    COALESCE(RCC5.ibk_indirecta_ytd_amt,0) ibk_indirecta_ytd_amt,
    COALESCE(RCC5.scotia_indirecta_ytd_amt,0) scotia_indirecta_ytd_amt,
    COALESCE(RCC5.cajas_finan_indirecta_ytd_amt,0) cajas_finan_indirecta_ytd_amt,
    COALESCE(RCC5.otros_indirecta_ytd_amt,0) otros_indirecta_ytd_amt,
    COALESCE(RCC5.var_scotia_indirecta_m1_amt,-99999999999) var_scotia_indirecta_m1_amt,
    COALESCE(RCC5.var_cajas_finan_indirecta_m1_amt,-99999999999) var_cajas_finan_indirecta_m1_amt,
    COALESCE(RCC5.var_otros_indirecta_m1_amt,-99999999999) var_otros_indirecta_m1_amt,
    COALESCE(RCC5.var_sistema_finan_m12_amt,-99999999999) var_sistema_finan_m12_amt,
    COALESCE(RCC5.var_bancos_m12_amt,-99999999999) var_bancos_m12_amt,
    COALESCE(RCC5.var_bbva_m12_amt,-99999999999) var_bbva_m12_amt,
    COALESCE(RCC5.var_bcp_m12_amt,-99999999999) var_bcp_m12_amt,
    COALESCE(RCC5.var_bif_m12_amt,-99999999999) var_bif_m12_amt,
    COALESCE(RCC5.var_ibk_m12_amt,-99999999999) var_ibk_m12_amt,
    COALESCE(RCC5.var_scotia_m12_amt,-99999999999) var_scotia_m12_amt,
    COALESCE(RCC5.var_cajas_finan_m12_amt,-99999999999) var_cajas_finan_m12_amt,
    COALESCE(RCC5.var_otros_m12_amt,-99999999999) var_otros_m12_amt,
    COALESCE(RCC5.var_sistema_finan_directa_m12_amt,-99999999999) var_sistema_finan_directa_m12_amt,
    COALESCE(RCC5.var_bancos_directa_m12_amt,-99999999999) var_bancos_directa_m12_amt,
    COALESCE(RCC5.var_bbva_directa_m12_amt,-99999999999) var_bbva_directa_m12_amt,
    COALESCE(RCC5.var_bcp_directa_m12_amt,-99999999999) var_bcp_directa_m12_amt,
    COALESCE(RCC5.var_bif_directa_m12_amt,-99999999999) var_bif_directa_m12_amt,
    COALESCE(RCC5.var_ibk_directa_m12_amt,-99999999999) var_ibk_directa_m12_amt,
    COALESCE(RCC5.var_scotia_directa_m12_amt,-99999999999) var_scotia_directa_m12_amt,
    COALESCE(RCC5.var_cajas_finan_directa_m12_amt,-99999999999) var_cajas_finan_directa_m12_amt,
    COALESCE(RCC5.var_otros_directa_m12_amt,-99999999999) var_otros_directa_m12_amt,
    COALESCE(RCC5.var_sistema_finan_indirecta_m12_amt,-99999999999) var_sistema_finan_indirecta_m12_amt,
    COALESCE(RCC5.var_bancos_indirecta_m12_amt,-99999999999) var_bancos_indirecta_m12_amt,
    COALESCE(RCC5.var_bbva_indirecta_m12_amt,-99999999999) var_bbva_indirecta_m12_amt,
    COALESCE(RCC5.var_bcp_indirecta_m12_amt,-99999999999) var_bcp_indirecta_m12_amt,
    COALESCE(RCC5.var_bif_indirecta_m12_amt,-99999999999) var_bif_indirecta_m12_amt,
    COALESCE(RCC5.var_ibk_indirecta_m12_amt,-99999999999) var_ibk_indirecta_m12_amt,
    COALESCE(RCC5.var_scotia_indirecta_m12_amt,-99999999999) var_scotia_indirecta_m12_amt,
    COALESCE(RCC5.var_cajas_finan_indirecta_m12_amt,-99999999999) var_cajas_finan_indirecta_m12_amt,
    COALESCE(RCC5.var_otros_indirecta_m12_amt,-99999999999) var_otros_indirecta_m12_amt,


    A.periodo_ejecucion
 
    FROM disc_comercial.hm_modelo_fuga_bn A   -- universo base

    LEFT JOIN  disc_comercial.hm_modelo_fuga_bn B
    ON A.NUMERORUC = B.NUMERORUC  AND A.periodo_ejecucion = SUBSTR(REPLACE(SUBSTR(CAST(date_add('month', -1, DATE_PARSE(CAST(B.periodo AS VARCHAR), '%Y%m')) AS VARCHAR), 1, 10), '-', ''), 1, 6)   
    
    LEFT JOIN disc_comercial.HM_VAR_SUNAT_RENIEC_FUGA_SG SR
    ON A.NUMERORUC = SR.num_ruc  AND A.periodo_ejecucion = SR.periodo_ejecucion  

    LEFT JOIN disc_comercial.HM_SALDO_VPC_IBK_TOTAL SLD
    ON A.NUMERORUC = SLD.num_ruc AND A.periodo_ejecucion = CAST(SLD.periodo  AS VARCHAR)
    
    LEFT JOIN disc_comercial.HM_PAGO_PLANILLAS_TOTAL PLA
    ON A.NUMERORUC = PLA.nro_doc_ord AND A.periodo_ejecucion = CAST(PLA.periodo_objetivo  AS VARCHAR)
    
    LEFT JOIN disc_comercial.HM_GESTION_VPCONNECT_HISTORICAS_BN VPC
    ON A.NUMERORUC = VPC.num_ruc  AND A.periodo_ejecucion = VPC.periodo_gestion

    LEFT JOIN (SELECT * FROM  disc_comercial.HM_BASE_SALDO_VARIABLES_12M_CONSOLIDADA_SG  WHERE num_ruc IN (SELECT DISTINCT NUMERORUC FROM disc_comercial.hm_modelo_fuga_bn) )  SF
    ON A.NUMERORUC = SF.num_ruc  AND A.periodo = SUBSTR(REPLACE(SUBSTR(CAST(date_add('month', +2, DATE_PARSE(CAST(SF.periodo_analisis AS VARCHAR), '%Y%m')) AS VARCHAR), 1, 10), '-', ''), 1, 6)   
    
    LEFT JOIN (SELECT * FROM  e_perm_aws.DS_DETALLE_CLIENTE_RCC_VPC WHERE num_ruc IN (SELECT DISTINCT NUMERORUC FROM disc_comercial.hm_modelo_fuga_bn) )  RCC1
    ON A.NUMERORUC = RCC1.num_ruc  AND A.periodo = RCC1.periodo_campania
    
    LEFT JOIN (SELECT * FROM  e_perm_aws.DS_HST_RCC_VPC WHERE num_ruc IN (SELECT DISTINCT NUMERORUC FROM disc_comercial.hm_modelo_fuga_bn) ) RCC2   
    ON A.NUMERORUC = RCC2.num_ruc  AND A.periodo = SUBSTR(REPLACE(SUBSTR(CAST(date_add('month', +2, DATE_PARSE(CAST(RCC2.cod_mes AS VARCHAR), '%Y%m')) AS VARCHAR), 1, 10), '-', ''), 1, 6)  

  
    LEFT JOIN (SELECT * FROM   e_perm_aws.DS_TENDENCIA_RCC_VPC WHERE num_ruc IN (SELECT DISTINCT NUMERORUC FROM disc_comercial.hm_modelo_fuga_bn) )  RCC3
    ON A.NUMERORUC = RCC3.num_ruc  AND A.periodo = SUBSTR(REPLACE(SUBSTR(CAST(date_add('month', +2, DATE_PARSE(CAST(RCC3.cod_mes AS VARCHAR), '%Y%m')) AS VARCHAR), 1, 10), '-', ''), 1, 6)  


    LEFT JOIN (SELECT * FROM  e_perm_aws.DS_VAR_SALDOS_VPC  WHERE num_ruc IN (SELECT DISTINCT NUMERORUC FROM disc_comercial.hm_modelo_fuga_bn) ) RCC5
    ON A.NUMERORUC = RCC5.num_ruc  AND A.periodo = SUBSTR(REPLACE(SUBSTR(CAST(date_add('month', +2, DATE_PARSE(CAST(RCC5.cod_mes AS VARCHAR), '%Y%m')) AS VARCHAR), 1, 10), '-', ''), 1, 6)  


   WHERE 1 = 1
            AND a.NUMERORUC IS NOT NULL
            AND a.NUMERORUC NOT LIKE ''
    """
)

In [ ]:
## SALDOS RCC VARIACION
df = wr.athena.read_sql_query("""
        SELECT periodo_campania	, count(1),  Sum(y_2m) y_2m, sum(y_3m) y_3m
        FROM disc_comercial.hm_kbr_base_modelo_fuga
        GROUP BY periodo_campania	
        ORDER BY periodo_campania	 DESC
    """,
    database="e_perm_aws",
    ctas_approach=False
)
print(df.shape)
df.head(10)

## UNIVERSO 1: VARIABLES REDUCIDAS